# 🗺️ Bandung AI Travel — Notebook 01: Recommendation Engine v2

**Capstone Project · Telkom University · Program Studi Data Science**

### Perubahan dari v1:
| # | Perubahan | Detail |
|---|---|---|
| 1 | **Hapus kategori Budaya** | Diganti 4 kategori: Alam, Kuliner, Wisata, Belanja |
| 2 | **Blacklist masjid & tempat ibadah** | 30+ keyword diblacklist dari nama destinasi |
| 3 | **Variety guarantee** | Jika user pilih >1 kategori, setiap kategori WAJIB ada minimal 1 destinasi |
| 4 | **Anti-spam interleaving** | Kandidat CBF di-interleave antar kategori sebelum masuk RL |
| 5 | **Variety reward dinaikkan** | `variety_bonus` 0.2 → 0.4, tambah `consecutive_penalty` |
| 6 | **Category swap enforcement** | Post-process RL: paksa swap jika ada kategori yang hilang |
| 7 | **Foursquare enrichment** | Tambah FSQ Places API (free) untuk rating yang lebih akurat |
| 8 | **Seed data diperluas** | 31 destinasi seed (dari 16), lebih representatif per kategori |


## 📦 CELL 1 — Setup & Dependencies

In [6]:
# ============================================================
# CELL 1 — Setup & Instalasi Dependencies
# ============================================================
import importlib.util, subprocess, sys

REQUIRED = {
    "requests": "requests", "pandas": "pandas", "numpy": "numpy",
    "sklearn": "scikit-learn", "matplotlib": "matplotlib",
    "seaborn": "seaborn", "tqdm": "tqdm",
}
missing = [pkg for mod, pkg in REQUIRED.items() if importlib.util.find_spec(mod) is None]
if missing:
    print(f"📦 Installing: {missing}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
print("✅ Semua dependency terpasang.")

import os, re, sys, json, math, time, pickle, random, urllib.parse, warnings
from pathlib import Path
from datetime import date
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings("ignore")

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

if Path.cwd().name == "notebooks":
    os.chdir("..")
for d in ["data/raw", "data/processed", "models", "notebooks"]:
    Path(d).mkdir(parents=True, exist_ok=True)

print(f"✅ Random seed: {RANDOM_SEED}")
print(f"📂 CWD: {Path.cwd()}")

# ── Konstanta kategori (Budaya DIHAPUS) ──────────────────────
# Perubahan v2: kategori Budaya dihapus karena banyak data
# tidak relevan (masjid, tempat ibadah) masuk lewat tag ini.
CATEGORY_ORDER  = ["Alam", "Kuliner", "Wisata", "Belanja"]
VALID_CATEGORIES = set(CATEGORY_ORDER)
print(f"✅ Kategori aktif: {CATEGORY_ORDER}")


✅ Semua dependency terpasang.
✅ Random seed: 42
📂 CWD: /kaggle/working
✅ Kategori aktif: ['Alam', 'Kuliner', 'Wisata', 'Belanja']


# ============================================================
# CELL 1 — Setup & Instalasi Dependencies
# ============================================================
# Auto-install: cek import; kalau ada yang missing, install otomatis sekali.
import importlib
import importlib.util
import subprocess
import sys

REQUIRED = {
    "requests": "requests",
    "pandas": "pandas",
    "numpy": "numpy",
    "sklearn": "scikit-learn",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "tqdm": "tqdm",
}
missing = [pkg for mod, pkg in REQUIRED.items()
           if importlib.util.find_spec(mod) is None]
if missing:
    print(f"📦 Installing missing packages: {missing}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    print("✅ Install selesai.")
else:
    print("✅ Semua dependency sudah terpasang.")

import os
import re
import sys
import json
import math
import time
import pickle
import random
import urllib.parse
import warnings
from pathlib import Path
from datetime import date
from collections import defaultdict, deque, Counter

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings("ignore")

# Reproducibility
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
print(f"✅ Random seed set: {RANDOM_SEED}")

# Pastikan working directory adalah ROOT proyek (bukan folder notebooks/)
if Path.cwd().name == "notebooks":
    os.chdir("..")
print(f"📂 CWD: {Path.cwd()}")

# Buat semua direktori yang dibutuhkan
for d in ["data/raw", "data/processed", "models", "notebooks"]:
    Path(d).mkdir(parents=True, exist_ok=True)

print("✅ Struktur folder berhasil dibuat")
print("📁 data/raw/        → hasil crawling mentah")
print("📁 data/processed/  → dataset bersih .csv")
print("📁 models/          → model .pkl tersimpan")


## 🌐 CELL 3 — Crawling: Overpass API + Foursquare Enrichment

> **Perubahan:** Budaya dihapus dari query, `place_of_worship` & `historic` dihapus, blacklist 30+ keyword (masjid, sekolah, bank, dll), tambah FSQ Places API untuk enrichment rating.

In [7]:
# ============================================================
# CELL 3 — Crawling: Overpass API + Foursquare FSQ Places
# ============================================================
# Dua sumber data FREE:
# 1. Overpass API (OSM)  — POI koordinat, nama, tags
# 2. Foursquare Places API (FSQ) — rating, kategori, tips
#    Daftar gratis di https://foursquare.com/developers
#    Free tier: 1000 req/hari — cukup untuk enrichment
#
# Improvement v2:
# - Budaya DIHAPUS dari query (hapus museum, memorial, dll)
# - place_of_worship DIHAPUS (sumber utama masjid masuk dataset)
# - historic DIHAPUS — terlalu noise
# - Tambah FSQ untuk enrichment rating & kategori yang lebih akurat

OVERPASS_ENDPOINTS = [
    "https://overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
    "https://overpass.private.coffee/api/interpreter",
    "https://overpass.openstreetmap.fr/api/interpreter",
]
OVERPASS_HEADERS = {
    "User-Agent": (
        "Bandung-AI-Travel-Capstone/2.0 "
        "(Telkom University; "
        "+https://github.com/Fall-Llihc/Bandung_AI_Travel-Capstone-Project)"
    ),
    "Accept": "application/json",
    "Accept-Encoding": "gzip, deflate",
}
BANDUNG_BBOX = "-7.2500,107.3500,-6.7500,107.9000"

# ── OSM tag → kategori (Budaya & place_of_worship DIHAPUS) ──
OSM_CATEGORY_MAP = {
    # Alam
    "viewpoint": "Alam", "natural": "Alam", "park": "Alam",
    "waterfall": "Alam", "forest": "Alam",
    # Kuliner
    "restaurant": "Kuliner", "cafe": "Kuliner",
    "food_court": "Kuliner", "fast_food": "Kuliner",
    # Wisata (theme park, attraction, dll)
    "theme_park": "Wisata", "attraction": "Wisata",
    "zoo": "Wisata", "aquarium": "Wisata", "amusement_ride": "Wisata",
    # Belanja
    "mall": "Belanja", "marketplace": "Belanja",
    "department_store": "Belanja",
}

# ── Query Overpass per kategori ──────────────────────────────
# PENTING: place_of_worship, historic, memorial DIHAPUS
# agar masjid/gereja/candi tidak masuk dataset wisata
CATEGORY_QUERIES = {
    "Alam": """
        node["tourism"="viewpoint"]({bbox});
        node["natural"="peak"]({bbox});
        node["natural"="waterfall"]({bbox});
        way["leisure"="park"]["name"~"."]({bbox});
        way["leisure"="nature_reserve"]({bbox});
        node["tourism"="picnic_site"]({bbox});
    """,
    "Kuliner": """
        node["amenity"="restaurant"]["name"~"."]({bbox});
        node["amenity"="cafe"]["name"~"."]({bbox});
        node["amenity"="food_court"]["name"~"."]({bbox});
        node["tourism"="restaurant"]["name"~"."]({bbox});
    """,
    "Wisata": """
        node["tourism"="theme_park"]({bbox});
        node["tourism"="attraction"]["name"~"."]({bbox});
        node["tourism"="zoo"]({bbox});
        node["tourism"="aquarium"]({bbox});
        way["tourism"="theme_park"]({bbox});
        way["leisure"="water_park"]({bbox});
        node["leisure"="miniature_golf"]({bbox});
    """,
    "Belanja": """
        node["shop"="mall"]["name"~"."]({bbox});
        node["amenity"="marketplace"]["name"~"."]({bbox});
        way["shop"="mall"]["name"~"."]({bbox});
        node["shop"="department_store"]["name"~"."]({bbox});
        node["shop"="clothes"]["name"~"."][~"^name$"~"Factory|Outlet|FO"]({bbox});
    """,
}

# ── BLACKLIST keywords — filter post-crawling ─────────────────
# Kata-kata ini di nama destinasi → HAPUS dari dataset
BLACKLIST_KEYWORDS = [
    # Tempat ibadah
    "masjid", "mushola", "musholah", "mosque", "gereja", "church",
    "vihara", "klenteng", "pura", "chapel", "cathedral",
    # Fasilitas umum bukan wisata
    "puskesmas", "rumah sakit", "klinik", "apotek", "kantor",
    "polsek", "polres", "koramil", "kodim",
    "sekolah", "sdn ", "smpn", "sman", "smkn", "sd ", "smp ", "sma ", "smk ",
    "universitas", "institut", "akademi", "kampus",
    "bank ", "atm ", "bri", "bni", "bca", "mandiri bank",
    "indomaret", "alfamart", "minimarket",
    "spbu", "pom bensin", "pertamina",
    "perumahan", "komplek", "ruko",
]

def is_blacklisted(name: str) -> bool:
    """Return True jika nama mengandung keyword blacklist."""
    name_lower = name.lower()
    return any(kw in name_lower for kw in BLACKLIST_KEYWORDS)


def query_overpass(category: str, bbox: str, endpoint: str, timeout: int = 60) -> list:
    q_template = CATEGORY_QUERIES.get(category, "")
    query = f"""
    [out:json][timeout:{timeout}];
    (
      {q_template.format(bbox=bbox)}
    );
    out center tags;
    """
    resp = requests.post(endpoint, data={"data": query},
                         headers=OVERPASS_HEADERS, timeout=timeout + 10)
    resp.raise_for_status()
    return resp.json().get("elements", [])


def parse_osm_element(el: dict, category: str) -> dict | None:
    tags = el.get("tags", {})
    name = (tags.get("name:id") or tags.get("name") or
            tags.get("name:en") or "").strip()
    if not name or len(name) < 3:
        return None

    # ── Filter blacklist ──────────────────────────────────────
    if is_blacklisted(name):
        return None

    # Koordinat
    if el["type"] == "node":
        lat, lng = el.get("lat"), el.get("lon")
    else:
        c = el.get("center", {})
        lat, lng = c.get("lat"), c.get("lon")
    if not lat or not lng:
        return None

    # GMaps URL
    query    = urllib.parse.quote(f"{name}, Bandung")
    gmaps_url = f"https://www.google.com/maps/search/?api=1&query={query}"

    # Tags semantik untuk TF-IDF
    semantic_tags = []
    for k, v in tags.items():
        if k in ("name", "name:id", "name:en", "source"): continue
        if k.startswith("addr:"): continue
        semantic_tags.append(v.lower().replace(" ", "_"))

    return {
        "id":          re.sub(r"[^a-z0-9]+", "-", name.lower()).strip("-"),
        "name":        name,
        "category":    category,
        "desc":        tags.get("description", ""),
        "ticket":      None,
        "duration":    None,
        "lat":         float(lat),
        "lng":         float(lng),
        "rating":      None,
        "tags":        semantic_tags[:8],
        "stay_detail": "",
        "gmaps_url":   gmaps_url,
        "source":      "osm",
    }


def run_overpass_crawl(verbose: bool = True) -> pd.DataFrame:
    all_records = []
    for category in CATEGORY_ORDER:
        if verbose: print(f"\n  Crawling OSM [{category}]...")
        elements = []
        for endpoint in OVERPASS_ENDPOINTS:
            try:
                elements = query_overpass(category, BANDUNG_BBOX, endpoint)
                if elements:
                    if verbose: print(f"    ✅ {endpoint.split('/')[2]}: {len(elements)} elements")
                    break
            except Exception as e:
                if verbose: print(f"    ⚠️  {endpoint.split('/')[2]}: {e}")
                time.sleep(2)

        parsed = []
        for el in elements:
            rec = parse_osm_element(el, category)
            if rec:
                parsed.append(rec)

        if verbose: print(f"    → {len(parsed)} valid destinations")
        all_records.extend(parsed)
        time.sleep(3)   # jangan spam Overpass

    df = pd.DataFrame(all_records)
    if df.empty:
        return df
    df = df.drop_duplicates(subset=["name", "category"])
    df = df.drop_duplicates(subset=["lat", "lng"])
    return df.reset_index(drop=True)


# ── Foursquare Places API enrichment ─────────────────────────
# Set FSQ_API_KEY = None jika tidak punya (akan skip enrichment)
# Daftar gratis: https://foursquare.com/developers
# Free: 1000 req/hari — cukup untuk 316 destinasi sekali pakai
FSQ_API_KEY = "fsq3Gdn78/pRDZ7uwDjTxtCAM1QIECRSPcmn7+hex469oo0="  # ← isi dengan key kamu, atau set via Kaggle Secrets
try:
    from kaggle_secrets import UserSecretsClient
    FSQ_API_KEY = UserSecretsClient().get_secret("FSQ_API_KEY")
    print("✅ FSQ_API_KEY dari Kaggle Secrets")
except Exception:
    pass

FSQ_CATEGORY_MAP = {
    # FSQ category ID → kategori proyek
    "4d4b7104d754a06370d81259": "Alam",       # Outdoors & Recreation
    "4d4b7105d754a06373d81259": "Kuliner",    # Food
    "4d4b7105d754a06374d81259": "Kuliner",    # Nightlife (cafe/bar)
    "4d4b7105d754a06376d81259": "Belanja",    # Shop & Service
    "4d4b7104d754a06368d81259": "Wisata",     # Arts & Entertainment
    "52e81612bcbc57f1066b7a21": "Wisata",     # Amusement Park
    "56aa371be4b08b9a8d57358a": "Alam",       # National Park
}

def fsq_search(name: str, lat: float, lng: float) -> dict | None:
    """Search satu destinasi di Foursquare Places API."""
    if not FSQ_API_KEY:
        return None
    try:
        url = "https://api.foursquare.com/v3/places/search"
        params = {"query": name, "ll": f"{lat},{lng}", "radius": 500, "limit": 1}
        resp = requests.get(url, headers={
            "Accept": "application/json",
            "Authorization": FSQ_API_KEY,
        }, params=params, timeout=8)
        resp.raise_for_status()
        results = resp.json().get("results", [])
        if results:
            return results[0]
    except Exception:
        pass
    return None


def enrich_with_fsq(df: pd.DataFrame, verbose: bool = True) -> pd.DataFrame:
    """Enrich rating dari Foursquare. Skip jika FSQ_API_KEY tidak ada."""
    if not FSQ_API_KEY:
        if verbose: print("  ⚠️  FSQ_API_KEY tidak ada — skip FSQ enrichment.")
        return df
    df = df.copy()
    enriched = 0
    for idx, row in tqdm(df.iterrows(), total=len(df), desc="  FSQ enrich"):
        if pd.notna(row.get("rating")) and row["rating"] > 0:
            continue
        result = fsq_search(row["name"], row["lat"], row["lng"])
        if result:
            fsq_rating = result.get("rating")  # 0–10
            if fsq_rating:
                df.at[idx, "rating"] = round(fsq_rating / 2, 1)  # konversi ke 0–5
                enriched += 1
        time.sleep(0.12)   # ~8 req/detik → aman di bawah rate limit FSQ
    if verbose: print(f"  ✅ FSQ: {enriched} destinasi di-enrich ratingnya.")
    return df


print("✅ Crawling functions siap.")
print(f"   Blacklist keywords: {len(BLACKLIST_KEYWORDS)} kata")
print(f"   Kategori aktif    : {CATEGORY_ORDER}")
print(f"   FSQ API Key       : {'✅ tersedia' if FSQ_API_KEY else '⚠️  tidak ada (skip enrichment)'}")


✅ Crawling functions siap.
   Blacklist keywords: 48 kata
   Kategori aktif    : ['Alam', 'Kuliner', 'Wisata', 'Belanja']
   FSQ API Key       : ✅ tersedia


## 🌱 CELL 4 — Seed Data + Run Crawling

> **Perubahan:** Semua entry Budaya dihapus. Seed diperluas ke 31 destinasi (8 Alam, 8 Kuliner, 8 Wisata, 7 Belanja).

In [8]:
# ============================================================
# CELL 4 — Seed Data + Run Crawling
# ============================================================
# Seed data: destinasi terpilih dengan data lengkap (termasuk
# ticket, duration, stay_detail yang tidak bisa di-crawl otomatis).
# Budaya DIHAPUS — semua entry Budaya dari versi lama dikonversi
# ke kategori yang lebih tepat atau dihapus.

SEED_DESTINATIONS = [
    # ===== ALAM (8) =====
    {"id": "kawah-putih", "name": "Kawah Putih", "category": "Alam",
     "desc": "Danau kawah vulkanik dengan air berwarna putih kehijauan di ketinggian 2430 mdpl",
     "ticket": 81000, "duration": 150, "lat": -7.1660, "lng": 107.4019, "rating": 4.6,
     "tags": ["sunrise", "fotogenik", "dingin", "kawah"],
     "stay_detail": "Parkir & jalan ke kawah 20', keliling kawah & foto 90', istirahat 20', kembali 20'"},
    {"id": "kebun-teh-rancabali", "name": "Kebun Teh Rancabali", "category": "Alam",
     "desc": "Hamparan kebun teh hijau di Ciwidey dengan udara sejuk dan pemandangan luas",
     "ticket": 25000, "duration": 105, "lat": -7.1432, "lng": 107.4106, "rating": 4.5,
     "tags": ["hijau", "fotogenik", "sejuk", "kebun"],
     "stay_detail": "Jalan ke area kebun 15', jalan-jalan & foto 60', duduk santai 30'"},
    {"id": "stone-garden-padalarang", "name": "Stone Garden Padalarang", "category": "Alam",
     "desc": "Taman batu purba dengan formasi karst unik di kawasan Citatah, Padalarang",
     "ticket": 15000, "duration": 105, "lat": -6.8323, "lng": 107.4709, "rating": 4.4,
     "tags": ["batu", "fotogenik", "karst", "alam"],
     "stay_detail": "Naik ke area batu 20', eksplorasi formasi & foto 60', istirahat 25'"},
    {"id": "tebing-keraton", "name": "Tebing Keraton", "category": "Alam",
     "desc": "Tebing dengan view hutan pinus Dago Pakar, populer untuk sunrise",
     "ticket": 15000, "duration": 120, "lat": -6.8359, "lng": 107.6630, "rating": 4.5,
     "tags": ["sunrise", "hiking", "tebing", "pinus"],
     "stay_detail": "Hiking ke tebing 30', menikmati view & foto 45', santai 25', turun 20'"},
    {"id": "tangkuban-perahu", "name": "Tangkuban Perahu", "category": "Alam",
     "desc": "Gunung berapi aktif dengan kawah ratu yang ikonik di utara Bandung",
     "ticket": 30000, "duration": 135, "lat": -6.7597, "lng": 107.6098, "rating": 4.5,
     "tags": ["gunung", "kawah", "legendaris", "vulkanik"],
     "stay_detail": "Parkir & jalan ke kawah 20', lihat kawah & foto 45', kawah domas 30', kembali 25', oleh-oleh 15'"},
    {"id": "situ-patenggang", "name": "Situ Patenggang", "category": "Alam",
     "desc": "Danau alami dikelilingi kebun teh di kawasan Ciwidey, romantis dan tenang",
     "ticket": 20000, "duration": 120, "lat": -7.1603, "lng": 107.3992, "rating": 4.4,
     "tags": ["danau", "romantis", "sejuk", "perahu"],
     "stay_detail": "Keliling danau 30', naik perahu 40', duduk santai & foto 30', jalan-jalan 20'"},
    {"id": "curug-malela", "name": "Curug Malela", "category": "Alam",
     "desc": "Air terjun lebar dijuluki Niagara Mini Indonesia di Bandung Barat",
     "ticket": 10000, "duration": 150, "lat": -6.9231, "lng": 107.3342, "rating": 4.6,
     "tags": ["air-terjun", "hiking", "segar", "alam"],
     "stay_detail": "Perjalanan & hiking ke curug 45', menikmati & foto 60', istirahat 25', kembali 20'"},
    {"id": "bukit-moko", "name": "Bukit Moko", "category": "Alam",
     "desc": "Puncak tertinggi di Bandung dengan panorama kota 360 derajat",
     "ticket": 20000, "duration": 120, "lat": -6.8198, "lng": 107.6898, "rating": 4.5,
     "tags": ["puncak", "panorama", "sunrise", "instagramable"],
     "stay_detail": "Hiking ke puncak 35', nikmati view & foto 50', turun 35'"},

    # ===== KULINER (8) =====
    {"id": "floating-market-lembang", "name": "Floating Market Lembang", "category": "Kuliner",
     "desc": "Pasar terapung dengan beragam jajanan tradisional Sunda di atas perahu",
     "ticket": 30000, "duration": 105, "lat": -6.8121, "lng": 107.6178, "rating": 4.3,
     "tags": ["jajanan", "keluarga", "fotogenik", "tradisional"],
     "stay_detail": "Keliling pasar & pilih 30', antri & beli 20', makan di perahu 35', foto 20'"},
    {"id": "jalan-braga", "name": "Jalan Braga Kuliner", "category": "Kuliner",
     "desc": "Kawasan heritage dengan deretan kafe dan restoran bergaya kolonial Belanda",
     "ticket": 0, "duration": 90, "lat": -6.9185, "lng": 107.6080, "rating": 4.3,
     "tags": ["heritage", "kafe", "kolonial", "santai"],
     "stay_detail": "Jalan-jalan di kawasan 20', pilih tempat makan 10', makan & santai 60'"},
    {"id": "cikole-lembang", "name": "Cikole Jayagiri Kuliner", "category": "Kuliner",
     "desc": "Kawasan wisata hutan pinus Lembang dengan warung-warung khas pegunungan",
     "ticket": 10000, "duration": 90, "lat": -6.8000, "lng": 107.6250, "rating": 4.2,
     "tags": ["pinus", "sejuk", "sate", "jagung-bakar"],
     "stay_detail": "Masuk area 10', pilih warung 10', pesan & tunggu 15', makan & santai 55'"},
    {"id": "sate-maranggi-haji-yetty", "name": "Sate Maranggi Hj. Yetty", "category": "Kuliner",
     "desc": "Sate maranggi legendaris khas Purwakarta-Cianjur yang terkenal di Bandung",
     "ticket": 0, "duration": 75, "lat": -6.9310, "lng": 107.6390, "rating": 4.7,
     "tags": ["sate", "legendaris", "khas-sunda", "enak"],
     "stay_detail": "Antri & pesan 15', tunggu sate dibakar 15', makan & santai 45'"},
    {"id": "warung-nasi-ampera", "name": "Warung Nasi Ampera", "category": "Kuliner",
     "desc": "Nasi Sunda lengkap dengan lauk-pauk otentik di kawasan Bandung kota",
     "ticket": 0, "duration": 60, "lat": -6.9200, "lng": 107.6100, "rating": 4.3,
     "tags": ["nasi-sunda", "murah", "otentik", "lauk-pauk"],
     "stay_detail": "Antri & pilih lauk 10', ambil nasi & minuman 5', makan 35', santai 10'"},
    {"id": "pasar-baru-bandung", "name": "Pasar Baru Kuliner", "category": "Kuliner",
     "desc": "Kawasan Pasar Baru dengan pilihan makanan kaki lima khas Bandung",
     "ticket": 0, "duration": 75, "lat": -6.9225, "lng": 107.6068, "rating": 4.1,
     "tags": ["kaki-lima", "murah", "batagor", "siomay"],
     "stay_detail": "Jalan-jalan cari makanan 20', beli jajanan 15', makan 30', lanjut keliling 10'"},
    {"id": "ikan-bakar-cianjur", "name": "Ikan Bakar Cianjur", "category": "Kuliner",
     "desc": "Restoran seafood dengan ikan bakar khas Sunda yang terkenal sejak 1980-an",
     "ticket": 0, "duration": 80, "lat": -6.9115, "lng": 107.6075, "rating": 4.5,
     "tags": ["ikan-bakar", "seafood", "legendaris", "sunda"],
     "stay_detail": "Pilih & pesan ikan 10', tunggu dibakar 20', makan & santai 50'"},
    {"id": "dago-pakar-resto", "name": "Dago Pakar Kuliner", "category": "Kuliner",
     "desc": "Deretan restoran di kawasan Dago Pakar dengan pemandangan kota Bandung",
     "ticket": 0, "duration": 90, "lat": -6.8450, "lng": 107.6420, "rating": 4.2,
     "tags": ["view-kota", "romantis", "restoran", "malam"],
     "stay_detail": "Pilih restoran & tempat duduk 10', pesan & tunggu 20', makan sambil lihat view 60'"},

    # ===== WISATA (8) =====
    {"id": "trans-studio-bandung", "name": "Trans Studio Bandung", "category": "Wisata",
     "desc": "Taman hiburan indoor terbesar di Bandung dengan berbagai wahana seru",
     "ticket": 150000, "duration": 270, "lat": -6.9246, "lng": 107.6376, "rating": 4.4,
     "tags": ["wahana", "keluarga", "indoor", "seru"],
     "stay_detail": "Antri masuk & orientasi 20', wahana utama x3 90', makan siang 45', wahana lanjutan 75', oleh-oleh 40'"},
    {"id": "kebun-binatang-bandung", "name": "Kebun Binatang Bandung", "category": "Wisata",
     "desc": "Kebun binatang klasik di pusat kota dengan koleksi satwa beragam",
     "ticket": 60000, "duration": 150, "lat": -6.9072, "lng": 107.6078, "rating": 4.1,
     "tags": ["binatang", "keluarga", "edukatif", "satwa"],
     "stay_detail": "Masuk & peta area 10', keliling zona satwa 90', foto satwa favorit 30', istirahat 20'"},
    {"id": "farmhouse-susu-lembang", "name": "Farmhouse Susu Lembang", "category": "Wisata",
     "desc": "Agrowisata bergaya Eropa di Lembang dengan atraksi susu segar dan hewan",
     "ticket": 35000, "duration": 120, "lat": -6.8214, "lng": 107.5956, "rating": 4.4,
     "tags": ["foto-eropa", "susu", "keluarga", "instagramable"],
     "stay_detail": "Masuk & foto venue 15', interaksi hewan 30', beli susu & foto 30', keliling area 25', oleh-oleh 20'"},
    {"id": "the-great-asia-africa", "name": "The Great Asia Africa", "category": "Wisata",
     "desc": "Destinasi wisata foto tematik dengan replika landmark Asia & Afrika",
     "ticket": 40000, "duration": 120, "lat": -6.9338, "lng": 107.4925, "rating": 4.3,
     "tags": ["foto-tematik", "instagramable", "landmark", "keluarga"],
     "stay_detail": "Masuk & orientasi 10', keliling zona Asia 40', foto landmark 40', zona Afrika 30'"},
    {"id": "dusun-bambu", "name": "Dusun Bambu Family Leisure Park", "category": "Wisata",
     "desc": "Ekowisata keluarga dengan konsep bambu di alam terbuka kawasan Lembang",
     "ticket": 25000, "duration": 180, "lat": -6.8083, "lng": 107.5497, "rating": 4.4,
     "tags": ["bambu", "ekowisata", "keluarga", "alam"],
     "stay_detail": "Masuk & peta 10', keliling area bambu 50', main wahana 40', makan siang 50', foto 30'"},
    {"id": "kampung-gajah", "name": "Kampung Gajah Wonderland", "category": "Wisata",
     "desc": "Resort wisata dengan kolam renang, mini golf, dan wahana keluarga",
     "ticket": 50000, "duration": 180, "lat": -6.8020, "lng": 107.5600, "rating": 4.0,
     "tags": ["kolam-renang", "wahana", "keluarga", "resort"],
     "stay_detail": "Ganti baju & masuk area 15', main air & wahana 90', istirahat & makan 45', main lagi 30'"},
    {"id": "maribaya-hot-spring", "name": "Maribaya Hot Spring", "category": "Wisata",
     "desc": "Sumber air panas alam dengan kolam renang dan waterpark di Lembang",
     "ticket": 20000, "duration": 150, "lat": -6.8167, "lng": 107.6500, "rating": 4.2,
     "tags": ["air-panas", "kolam", "alam", "relaksasi"],
     "stay_detail": "Masuk & ganti 15', rendam air panas 60', main waterpark 45', bilas & ganti 30'"},
    {"id": "wisata-edukasi-susu-lembang", "name": "Wisata Edukasi Susu Lembang", "category": "Wisata",
     "desc": "Agrowisata edukasi cara peternakan sapi dan proses pembuatan susu segar",
     "ticket": 25000, "duration": 90, "lat": -6.8300, "lng": 107.6100, "rating": 4.3,
     "tags": ["edukasi", "sapi", "susu", "keluarga"],
     "stay_detail": "Tur peternakan 30', proses susu 20', cicipi susu segar 15', oleh-oleh 25'"},

    # ===== BELANJA (7) =====
    {"id": "cibaduyut-shoe-street", "name": "Cibaduyut Sentra Sepatu", "category": "Belanja",
     "desc": "Pusat industri dan penjualan sepatu kulit buatan tangan khas Bandung",
     "ticket": 0, "duration": 90, "lat": -6.9651, "lng": 107.5917, "rating": 4.2,
     "tags": ["sepatu", "kulit", "murah", "produk-lokal"],
     "stay_detail": "Keliling toko 30', tawar-menawar & coba sepatu 40', beli & bayar 20'"},
    {"id": "pasar-baru-trade-center", "name": "Pasar Baru Trade Center", "category": "Belanja",
     "desc": "Pusat perbelanjaan tekstil dan fashion terbesar di Bandung sejak era kolonial",
     "ticket": 0, "duration": 120, "lat": -6.9225, "lng": 107.6068, "rating": 4.1,
     "tags": ["tekstil", "fashion", "murah", "grosir"],
     "stay_detail": "Orientasi lantai 10', belanja tekstil & baju 70', tawar-menawar 25', checkout 15'"},
    {"id": "cihampelas-walk", "name": "Cihampelas Walk (Ciwalk)", "category": "Belanja",
     "desc": "Mall semi outdoor dengan konsep unik di Jalan Cihampelas yang ikonik",
     "ticket": 0, "duration": 120, "lat": -6.8990, "lng": 107.6053, "rating": 4.3,
     "tags": ["semi-outdoor", "mall", "jeans", "fashion"],
     "stay_detail": "Keliling area outdoor 20', belanja toko jeans 50', makan 30', foto spot ikonik 20'"},
    {"id": "factory-outlet-dago", "name": "Factory Outlet Dago", "category": "Belanja",
     "desc": "Deretan factory outlet premium di Jalan Dago dengan merek lokal dan internasional",
     "ticket": 0, "duration": 105, "lat": -6.8800, "lng": 107.6130, "rating": 4.2,
     "tags": ["factory-outlet", "premium", "diskon", "merek"],
     "stay_detail": "Pilih FO yang dituju 10', belanja & coba baju 70', checkout 15', makan ringan 10'"},
    {"id": "cibaduyut-leather-village", "name": "Kampung Tas Cibaduyut", "category": "Belanja",
     "desc": "Pusat kerajinan tas kulit buatan tangan dengan harga produsen langsung",
     "ticket": 0, "duration": 75, "lat": -6.9640, "lng": 107.5920, "rating": 4.1,
     "tags": ["tas", "kulit", "handmade", "murah"],
     "stay_detail": "Keliling toko tas 25', pilih & coba tas 30', tawar & beli 20'"},
    {"id": "bandung-indah-plaza", "name": "Bandung Indah Plaza (BIP)", "category": "Belanja",
     "desc": "Mall pusat kota Bandung yang lengkap dengan brand lokal dan internasional",
     "ticket": 0, "duration": 120, "lat": -6.9175, "lng": 107.6079, "rating": 4.2,
     "tags": ["mall", "lengkap", "pusat-kota", "brand"],
     "stay_detail": "Orientasi 10', window shopping & belanja 70', makan 30', jajanan 10'"},
    {"id": "rumah-mode-festival", "name": "Rumah Mode Festival", "category": "Belanja",
     "desc": "Pusat batik dan fashion lokal Bandung dengan koleksi distro dan clothing",
     "ticket": 0, "duration": 90, "lat": -6.8953, "lng": 107.6082, "rating": 4.0,
     "tags": ["batik", "distro", "fashion-lokal", "clothing"],
     "stay_detail": "Masuk & orientasi 10', belanja distro & batik 60', tawar 15', checkout 5'"},
]

print(f"✅ Seed destinations: {len(SEED_DESTINATIONS)}")
from collections import Counter
counts = Counter(d["category"] for d in SEED_DESTINATIONS)
for cat, n in sorted(counts.items()):
    print(f"   {cat:<10}: {n}")

# ── Run crawling ──────────────────────────────────────────────
print("\n🌐 Memulai crawling Overpass API...")
df_osm = run_overpass_crawl(verbose=True)
print(f"\n📊 Hasil crawling OSM: {len(df_osm)} destinasi")
if not df_osm.empty:
    print(df_osm["category"].value_counts().to_dict())

# Simpan hasil crawling mentah
if not df_osm.empty:
    df_osm.to_csv("data/raw/osm_raw.csv", index=False)
    print("💾 data/raw/osm_raw.csv")

# ── Gabungkan seed + OSM ──────────────────────────────────────
df_seed = pd.DataFrame(SEED_DESTINATIONS)
if not df_osm.empty:
    df_combined = pd.concat([df_seed, df_osm], ignore_index=True)
else:
    df_combined = df_seed.copy()
    print("⚠️  OSM crawling gagal — menggunakan seed data saja")

# De-duplicate: seed lebih diprioritaskan (sudah ada duluan)
df_combined = df_combined.drop_duplicates(subset=["id"], keep="first")

# Aplikasikan blacklist sekali lagi ke semua data (termasuk seed)
before = len(df_combined)
df_combined = df_combined[~df_combined["name"].apply(is_blacklisted)].reset_index(drop=True)
removed = before - len(df_combined)
if removed:
    print(f"🚫 Dihapus karena blacklist: {removed} destinasi")

print(f"\n📦 Total sebelum cleaning: {len(df_combined)}")
print(df_combined["category"].value_counts().to_dict())


✅ Seed destinations: 31
   Alam      : 8
   Belanja   : 7
   Kuliner   : 8
   Wisata    : 8

🌐 Memulai crawling Overpass API...

  Crawling OSM [Alam]...
    ✅ overpass-api.de: 1362 elements
    → 840 valid destinations

  Crawling OSM [Kuliner]...
    ✅ overpass-api.de: 1369 elements
    → 1365 valid destinations

  Crawling OSM [Wisata]...
    ✅ overpass-api.de: 71 elements
    → 69 valid destinations

  Crawling OSM [Belanja]...
    ⚠️  overpass-api.de: 429 Client Error: Too Many Requests for url: https://overpass-api.de/api/interpreter
    ⚠️  overpass.kumi.systems: HTTPSConnectionPool(host='overpass.kumi.systems', port=443): Read timed out. (read timeout=70)
    ✅ overpass.private.coffee: 63 elements
    → 62 valid destinations

📊 Hasil crawling OSM: 1980 destinasi
{'Kuliner': 1107, 'Alam': 745, 'Wisata': 68, 'Belanja': 60}
💾 data/raw/osm_raw.csv

📦 Total sebelum cleaning: 1991
{'Kuliner': 1104, 'Alam': 752, 'Wisata': 69, 'Belanja': 66}


## 🧹 CELL 5 — Data Cleaning

> **Perubahan:** `VALID_CATEGORIES` tidak mengandung Budaya. Default values untuk 4 kategori. Re-apply blacklist sebagai failsafe.

In [9]:
# ============================================================
# CELL 5 — Data Cleaning & Validation
# ============================================================
# Perubahan v2:
# - VALID_CATEGORIES tidak mengandung "Budaya"
# - DEFAULT values disesuaikan untuk 4 kategori
# - Filter tambahan: hapus duplikat koordinat, nama terlalu pendek

# Default fallback jika nilai NaN
DEFAULT_TICKET   = {"Alam": 20000, "Kuliner": 30000, "Wisata": 50000, "Belanja": 0}
DEFAULT_DURATION = {"Alam": 120,   "Kuliner": 75,    "Wisata": 135,   "Belanja": 105}
DEFAULT_RATING   = 4.2


def clean_destinations(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # 0. Pastikan semua kolom ada
    for col in ("id","name","category","desc","ticket","duration",
                "rating","tags","lat","lng","gmaps_url","stay_detail"):
        if col not in df.columns:
            df[col] = pd.NA

    # 1. Rename lon → lng
    if "lon" in df.columns and "lng" not in df.columns:
        df = df.rename(columns={"lon": "lng"})
    elif "lon" in df.columns:
        df["lng"] = df["lng"].fillna(df["lon"])
        df = df.drop(columns=["lon"])

    # 2. Filter kategori valid (Budaya tidak ada di VALID_CATEGORIES)
    before = len(df)
    df = df[df["category"].isin(VALID_CATEGORIES)].copy()
    removed = before - len(df)
    if removed:
        print(f"  🗑️  Dihapus kategori tidak valid (termasuk Budaya): {removed}")

    # 3. Filter nama terlalu pendek atau angka saja
    df = df[df["name"].str.len() >= 3]
    df = df[~df["name"].str.match(r"^\d+$")]

    # 4. Re-apply blacklist (failsafe)
    before = len(df)
    df = df[~df["name"].apply(is_blacklisted)]
    removed = before - len(df)
    if removed:
        print(f"  🚫 Blacklist filter: {removed} dihapus")

    # 5. Fill missing values per kategori
    for cat in VALID_CATEGORIES:
        mask = df["category"] == cat
        if mask.sum() == 0:
            continue
        med_ticket = df.loc[mask, "ticket"].median()
        df.loc[mask, "ticket"] = df.loc[mask, "ticket"].fillna(
            med_ticket if pd.notna(med_ticket) else DEFAULT_TICKET[cat]
        )
        mean_rating = df.loc[mask, "rating"].mean()
        df.loc[mask, "rating"] = df.loc[mask, "rating"].fillna(
            mean_rating if pd.notna(mean_rating) else DEFAULT_RATING
        )
        med_dur = df.loc[mask, "duration"].median()
        df.loc[mask, "duration"] = df.loc[mask, "duration"].fillna(
            med_dur if pd.notna(med_dur) else DEFAULT_DURATION[cat]
        )

    df["desc"]       = df["desc"].fillna(df["category"].apply(lambda c: f"Destinasi wisata {c} di Bandung"))
    df["tags"]       = df["tags"].apply(lambda x: x if isinstance(x, list) else [])
    df["stay_detail"] = df["stay_detail"].fillna("")
    df["gmaps_url"]  = df["gmaps_url"].fillna(
        df["name"].apply(lambda n: f"https://www.google.com/maps/search/?api=1&query={urllib.parse.quote(n+', Bandung')}")
    )

    # 6. Type casting
    df["ticket"]   = df["ticket"].astype(float).round().astype(int)
    df["duration"] = df["duration"].astype(float).round().astype(int).clip(lower=30)
    df["rating"]   = df["rating"].astype(float).clip(1.0, 5.0).round(1)
    df["lat"]      = df["lat"].astype(float)
    df["lng"]      = df["lng"].astype(float)

    # 7. Drop rows dengan koordinat invalid
    df = df[df["lat"].between(-8.0, -6.5) & df["lng"].between(107.0, 108.5)]

    # 8. Buat ID slug unik
    df["id"] = df["name"].apply(lambda n: re.sub(r"[^a-z0-9]+", "-", n.lower()).strip("-"))
    df = df.drop_duplicates(subset=["id"], keep="first")

    # 9. De-duplikasi koordinat (radius ~100m)
    df["lat_r"] = df["lat"].round(3)
    df["lng_r"] = df["lng"].round(3)
    df = df.drop_duplicates(subset=["lat_r","lng_r"], keep="first")
    df = df.drop(columns=["lat_r","lng_r"])

    return df.reset_index(drop=True)


# ── Enrich FSQ dulu sebelum cleaning ─────────────────────────
print("🔍 FSQ enrichment...")
df_enriched = enrich_with_fsq(df_combined, verbose=True)

# ── Cleaning ──────────────────────────────────────────────────
print("\n🧹 Cleaning...")
df_clean = clean_destinations(df_enriched)

print(f"\n✅ Dataset bersih: {len(df_clean)} destinasi")
print(df_clean["category"].value_counts().to_dict())
print(f"\n📊 Statistik:")
print(f"   Harga   : Rp {df_clean['ticket'].min():,} – Rp {df_clean['ticket'].max():,}")
print(f"   Rating  : {df_clean['rating'].min():.1f} – {df_clean['rating'].max():.1f}")
print(f"   Durasi  : {df_clean['duration'].min()} – {df_clean['duration'].max()} menit")

df_clean.to_csv("data/processed/destinations.csv", index=False)
with open("data/last_updated.txt", "w") as f:
    f.write(str(date.today()))
print("\n💾 data/processed/destinations.csv")
print("💾 data/last_updated.txt")


🔍 FSQ enrichment...


  FSQ enrich: 100%|██████████| 1991/1991 [34:25<00:00,  1.04s/it]  


  ✅ FSQ: 0 destinasi di-enrich ratingnya.

🧹 Cleaning...

✅ Dataset bersih: 1476 destinasi
{'Alam': 727, 'Kuliner': 646, 'Wisata': 61, 'Belanja': 42}

📊 Statistik:
   Harga   : Rp 0 – Rp 150,000
   Rating  : 4.0 – 4.7
   Durasi  : 60 – 270 menit

💾 data/processed/destinations.csv
💾 data/last_updated.txt


# ============================================================
# CELL 5 — Data Cleaning & Validation
# ============================================================
VALID_CATEGORIES = {"Alam", "Kuliner", "Budaya", "Wisata", "Belanja"}

# Default fallback jika semua nilai kategori NaN
DEFAULT_TICKET = {"Alam": 20000, "Kuliner": 35000, "Budaya": 10000, "Wisata": 50000, "Belanja": 0}
DEFAULT_DURATION = {"Alam": 120, "Kuliner": 75, "Budaya": 80, "Wisata": 135, "Belanja": 105}
DEFAULT_RATING = 4.2


def clean_destinations(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # 0. Pastikan semua kolom wajib ada (idempotent)
    for col in ("id", "name", "category", "desc", "ticket", "duration",
                "rating", "tags", "lat", "lng", "gmaps_url", "stay_detail"):
        if col not in df.columns:
            df[col] = pd.NA

    # 1. Rename lon → lng (jika masih ada)
    if "lon" in df.columns and "lng" not in df.columns:
        df = df.rename(columns={"lon": "lng"})
    elif "lon" in df.columns and "lng" in df.columns:
        df["lng"] = df["lng"].fillna(df["lon"])
        df = df.drop(columns=["lon"])

    # 2. Filter kategori valid
    df = df[df["category"].isin(VALID_CATEGORIES)].copy()

    # 3. Fill missing — ticket, rating, duration per kategori
    for cat in VALID_CATEGORIES:
        mask = df["category"] == cat
        med_ticket = df.loc[mask, "ticket"].median()
        if pd.isna(med_ticket):
            med_ticket = DEFAULT_TICKET[cat]
        df.loc[mask, "ticket"] = df.loc[mask, "ticket"].fillna(med_ticket)

        mean_rating = df.loc[mask, "rating"].mean()
        if pd.isna(mean_rating):
            mean_rating = DEFAULT_RATING
        df.loc[mask, "rating"] = df.loc[mask, "rating"].fillna(mean_rating)

        med_dur = df.loc[mask, "duration"].median()
        if pd.isna(med_dur):
            med_dur = DEFAULT_DURATION[cat]
        df.loc[mask, "duration"] = df.loc[mask, "duration"].fillna(med_dur)

    df["desc"] = df["desc"].fillna(df["category"].apply(lambda c: f"Destinasi wisata {c} di Bandung"))
    df["tags"] = df["tags"].apply(lambda x: x if isinstance(x, list) else [])
    df["stay_detail"] = df["stay_detail"].fillna("")

    # 4. Type casting
    df["ticket"] = df["ticket"].astype(float).round().astype(int)
    df["duration"] = df["duration"].astype(float).round().astype(int)
    df["rating"] = df["rating"].astype(float).clip(1.0, 5.0)
    df["lat"] = df["lat"].astype(float)
    df["lng"] = df["lng"].astype(float)

    # 5. Validasi geografis & nama
    df = df[df["lat"].between(-7.4, -6.6)]
    df = df[df["lng"].between(107.2, 108.2)]
    df = df[df["name"].notna() & (df["name"].astype(str).str.len() >= 3)]

    # 6. Generate slug id jika kosong
    df["id"] = df.apply(
        lambda r: r["id"] if pd.notna(r["id"]) and str(r["id"]).strip() else slugify(r["name"]),
        axis=1,
    )

    # 7. Sort by category + rating desc
    df = df.sort_values(["category", "rating"], ascending=[True, False]).reset_index(drop=True)

    # 8. Pastikan gmaps_url ada
    if "gmaps_url" not in df.columns or df["gmaps_url"].isna().any():
        df["gmaps_url"] = df["gmaps_url"].fillna(df["name"].apply(generate_gmaps_url))

    return df


df_clean = clean_destinations(df_enriched)

# Assert minimal 50 destinasi & semua kategori
assert len(df_clean) >= 50, f"Dataset minimal 50 destinasi, dapat {len(df_clean)}"
assert set(df_clean["category"].unique()) == VALID_CATEGORIES, \
    f"Kategori tidak lengkap: {df_clean['category'].unique()}"

# Pastikan 16 seed destinations tetap ada
seed_ids = {d["id"] for d in SEED_DESTINATIONS}
present = seed_ids & set(df_clean["id"].tolist())
assert len(present) == len(seed_ids), f"Seed hilang: {seed_ids - present}"

print(f"✅ Dataset valid: {len(df_clean)} destinasi")
print("\nDistribusi per kategori:")
print(df_clean.groupby("category").size())

# Simpan
df_clean.to_csv("data/processed/destinations.csv", index=False)
with open("data/last_updated.txt", "w") as f:
    f.write(str(date.today()))

print("\n💾 data/processed/destinations.csv tersimpan")
print(f"💾 data/last_updated.txt = {date.today()}")
print("\nSample 5 destinasi teratas (rating tertinggi per kategori):")
print(df_clean.head().to_string(index=False))


## ⚙️ Fase 2: Feature Engineering

Mengubah atribut destinasi menjadi vektor numerik untuk:
1. **CBF Model** — menghitung cosine similarity antar destinasi
2. **RL Agent** — representasi state/action sebagai vektor

| Fitur | Tipe | Keterangan |
|---|---|---|
| `category_*` | One-hot (5 kolom) | Alam, Kuliner, Budaya, Wisata, Belanja |
| `rating_norm` | float [0,1] | Rating dinormalisasi |
| `ticket_norm` | float [0,1] | Harga tiket dinormalisasi (log-scale) |
| `duration_norm` | float [0,1] | Durasi kunjungan dinormalisasi |
| `lat_norm`, `lng_norm` | float [0,1] | Koordinat dinormalisasi |
| `tags_tfidf_*` | float [0,1] (n kolom) | TF-IDF dari kolom tags |

Total dimensi: 5 (category) + 5 (numerik) + n_tfidf (tags) ≈ 20–30 dimensi.

## ⚙️ CELL 7 — Feature Engineering

> **Perubahan:** `CATEGORY_ORDER` 4 kolom (tanpa Budaya). Bobot kategori ×2.0 (dari ×1.0) agar similarity antar kategori berbeda lebih rendah → mendukung variety.

In [13]:
# ============================================================
# CELL 7 — Feature Engineering
# ============================================================
# Perubahan v2:
# - CATEGORY_ORDER hanya 4 (Budaya dihapus)
# - Feature matrix shape berubah: N × (4 + numeric + tfidf)

df_feat = df_clean.copy()

# (a) One-hot kategori — 4 kolom (Budaya dihapus)
# CATEGORY_ORDER = ["Alam","Kuliner","Wisata","Belanja"] — dari Cell 1
category_onehot = np.zeros((len(df_feat), len(CATEGORY_ORDER)), dtype=float)
for i, cat in enumerate(df_feat["category"].tolist()):
    if cat in CATEGORY_ORDER:
        category_onehot[i, CATEGORY_ORDER.index(cat)] = 1.0

# (b) Normalisasi numerik
df_feat["ticket_log"] = np.log1p(df_feat["ticket"].astype(float))
numeric_cols = ["ticket_log", "rating", "duration", "lat", "lng"]
scaler = MinMaxScaler()
numeric_normalized = scaler.fit_transform(df_feat[numeric_cols].values)

# (c) TF-IDF dari tags
df_feat["tags_str"] = df_feat["tags"].apply(
    lambda x: " ".join(x) if isinstance(x, list) else str(x)
)
tfidf = TfidfVectorizer(max_features=20, min_df=1)
tfidf_matrix = tfidf.fit_transform(df_feat["tags_str"]).toarray()

tfidf_scaler = None
if tfidf_matrix.shape[1] > 0 and tfidf_matrix.max() > 0:
    tfidf_scaler = MinMaxScaler()
    tfidf_matrix = tfidf_scaler.fit_transform(tfidf_matrix)

# (d) Gabungkan — bobot kategori lebih tinggi agar similarity
#     antar kategori berbeda lebih rendah (penting untuk variety)
feature_matrix = np.hstack([
    category_onehot * 2.0,   # bobot ×2 agar kategori lebih berpengaruh
    numeric_normalized,
    tfidf_matrix * 0.5
])

print(f"✅ Feature matrix: {feature_matrix.shape}")
print(f"   {category_onehot.shape[1]} kategori (×2.0) + "
      f"{numeric_normalized.shape[1]} numerik + "
      f"{tfidf_matrix.shape[1]} TF-IDF (×0.5)")

np.save("data/processed/feature_matrix.npy", feature_matrix)

le_category = LabelEncoder()
le_category.fit(CATEGORY_ORDER)

encoders = {
    "label_encoder_category": le_category,
    "scaler":         scaler,
    "tfidf":          tfidf,
    "tfidf_scaler":   tfidf_scaler,
    "feature_cols":   numeric_cols,
    "category_order": CATEGORY_ORDER,       # ["Alam","Kuliner","Wisata","Belanja"]
    "n_category_dims": category_onehot.shape[1],
    "n_numeric_dims":  numeric_normalized.shape[1],
    "n_tfidf_dims":    tfidf_matrix.shape[1],
}
with open("models/label_encoders.pkl", "wb") as f:
    pickle.dump(encoders, f)
with open("models/scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

print("💾 feature_matrix.npy | label_encoders.pkl | scaler.pkl")


✅ Feature matrix: (1476, 29)
   4 kategori (×2.0) + 5 numerik + 20 TF-IDF (×0.5)
💾 feature_matrix.npy | label_encoders.pkl | scaler.pkl


# ============================================================
# CELL 7 — Feature Engineering
# ============================================================
df_feat = df_clean.copy()

# (a) One-hot encoding kategori (urutan tetap)
CATEGORY_ORDER = ["Alam", "Kuliner", "Budaya", "Wisata", "Belanja"]
le_category = LabelEncoder()
le_category.fit(CATEGORY_ORDER)

category_onehot = np.zeros((len(df_feat), len(CATEGORY_ORDER)), dtype=float)
for i, cat in enumerate(df_feat["category"].tolist()):
    category_onehot[i, CATEGORY_ORDER.index(cat)] = 1.0

# (b) Normalisasi numerik (ticket → log-scale dulu)
df_feat["ticket_log"] = np.log1p(df_feat["ticket"].astype(float))

numeric_cols = ["ticket_log", "rating", "duration", "lat", "lng"]
scaler = MinMaxScaler()
numeric_normalized = scaler.fit_transform(df_feat[numeric_cols].values)

# (c) TF-IDF dari tags
df_feat["tags_str"] = df_feat["tags"].apply(
    lambda x: " ".join(x) if isinstance(x, list) else str(x)
)
tfidf = TfidfVectorizer(max_features=20, min_df=1)
tfidf_matrix = tfidf.fit_transform(df_feat["tags_str"]).toarray()

# (c.1) MinMax normalize TF-IDF agar magnitude-nya sebanding dengan one-hot/numeric
# (raw TF-IDF L2-normalized → kolom kecil; ini meningkatkan kontribusi dim tags
#  pada cosine similarity tanpa mendominasi kolom kategori one-hot)
tfidf_scaler = MinMaxScaler()
if tfidf_matrix.shape[1] > 0 and tfidf_matrix.max() > 0:
    tfidf_matrix = tfidf_scaler.fit_transform(tfidf_matrix)
else:
    tfidf_scaler = None

# (d) Gabungkan
feature_matrix = np.hstack([category_onehot, numeric_normalized, tfidf_matrix])
print(f"✅ Feature matrix shape: {feature_matrix.shape}")
print(f"   - {category_onehot.shape[1]} kolom kategori")
print(f"   - {numeric_normalized.shape[1]} kolom numerik")
print(f"   - {tfidf_matrix.shape[1]} kolom TF-IDF")

# Simpan feature matrix
np.save("data/processed/feature_matrix.npy", feature_matrix)

# Simpan scaler & encoders
encoders = {
    "label_encoder_category": le_category,
    "scaler": scaler,
    "tfidf": tfidf,
    "tfidf_scaler": tfidf_scaler,
    "feature_cols": numeric_cols,
    "category_order": CATEGORY_ORDER,
    "n_category_dims": category_onehot.shape[1],
    "n_numeric_dims": numeric_normalized.shape[1],
    "n_tfidf_dims": tfidf_matrix.shape[1],
}
with open("models/label_encoders.pkl", "wb") as f:
    pickle.dump(encoders, f)

with open("models/scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

print("\n💾 data/processed/feature_matrix.npy")
print("💾 models/label_encoders.pkl")
print("💾 models/scaler.pkl")


## 🔍 CELL 9 — Content-Based Filtering + Variety-Aware Recommend

> **Perubahan kunci:**
> 1. `recommend()` per-kategori guarantee: setiap kategori yang diminta user dapat `ceil(top_n/len(cats))` slot minimum
> 2. `interleave_by_category()`: hasil diurutkan round-robin antar kategori sebelum masuk RL


In [19]:
# ============================================================
# CELL 9 — Content-Based Filtering + Variety-Aware Recommend
# ============================================================
# Perubahan v2:
# - recommend() wajib guarantee minimal 1 destinasi per kategori
#   yang diminta user (jika user pilih >1 kategori)
# - Interleaving: hasil diurutkan selang-seling antar kategori
#   agar tidak spam 1 kategori dari awal sampai akhir

def haversine_km(lat1, lng1, lat2, lng2) -> float:
    R = 6371.0
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    a = (math.sin(math.radians(lat2-lat1)/2)**2 +
         math.cos(phi1)*math.cos(phi2)*math.sin(math.radians(lng2-lng1)/2)**2)
    return 2*R*math.asin(math.sqrt(a))


class ContentBasedFilter:

    def __init__(self, feature_matrix, destinations_df, encoders):
        self.feature_matrix   = feature_matrix
        self.df               = destinations_df.reset_index(drop=True)
        self.encoders         = encoders
        self.similarity_matrix = None

    def fit(self):
        self.similarity_matrix = cosine_similarity(self.feature_matrix)
        print(f"✅ CBF fitted. Similarity matrix: {self.similarity_matrix.shape}")
        return self

    def _category_dims(self): return self.encoders.get("n_category_dims", 4)
    def _numeric_start(self): return self._category_dims()
    def _numeric_end(self):   return self._numeric_start() + self.encoders.get("n_numeric_dims", 5)

    def build_user_profile(self, categories, budget=None):
        mask = (self.df["category"].isin(categories).values
                if categories else np.ones(len(self.df), bool))
        if mask.sum() == 0:
            mask = np.ones(len(self.df), bool)
        profile = self.feature_matrix[mask].mean(axis=0)
        if budget is not None and budget > 0:
            try:
                sc      = self.encoders["scaler"]
                cols    = self.encoders["feature_cols"]
                t_idx   = cols.index("ticket_log")
                dummy   = np.zeros((1, len(cols)))
                for j, col in enumerate(cols):
                    dummy[0,j] = (math.log1p(budget) if col=="ticket_log"
                                  else self.feature_matrix[:,self._numeric_start()+j].mean())
                norm    = sc.transform(dummy)[0]
                ns      = self._numeric_start()
                profile = profile.copy()
                profile[ns + t_idx] = norm[t_idx]
            except Exception:
                pass
        return profile

    def recommend(self, categories=None, budget=None, max_km=None,
                  home_lat=-6.9215, home_lng=107.6071, top_n=60):
        """
        Kembalikan top-N kandidat.

        Jaminan v2:
        1. Setiap kategori yang diminta user WAJIB diwakili minimal
           ceil(top_n / len(categories)) kandidat.
        2. Hasil di-interleave antar kategori sehingga urutan tidak
           monoton (tidak spam 1 kategori beruntun).
        """
        cats = categories or []

        # ── Filter hard constraints ───────────────────────────
        mask_cat    = self.df["category"].isin(cats) if cats else pd.Series([True]*len(self.df))
        mask_budget = (self.df["ticket"] <= budget) if budget is not None else pd.Series([True]*len(self.df))
        if max_km is not None:
            dist     = self.df.apply(lambda r: haversine_km(home_lat,home_lng,r["lat"],r["lng"]), axis=1)
            mask_km  = dist <= max_km
        else:
            mask_km  = pd.Series([True]*len(self.df))

        mask = mask_cat & mask_budget & mask_km
        # Fallback: relax budget/km jika kandidat terlalu sedikit
        if mask.sum() < max(len(cats)*2, 5):
            mask = mask_cat
        if mask.sum() == 0:
            mask = pd.Series([True]*len(self.df))

        # ── Cosine similarity score ───────────────────────────
        profile = self.build_user_profile(cats, budget)
        scores  = cosine_similarity([profile], self.feature_matrix)[0]
        scores  = scores * mask.values.astype(float)

        # ── Per-kategori guarantee ────────────────────────────
        # Jika user memilih >1 kategori: ambil top per-kategori dulu,
        # baru gabungkan, baru isi sisanya dari global top score.
        if len(cats) > 1:
            per_cat_n = max(3, top_n // len(cats))  # minimal 3 per kategori
            result_idx = []
            seen       = set()

            for cat in cats:
                cat_mask  = (self.df["category"] == cat).values & mask.values.astype(bool)
                cat_scores = scores * cat_mask.astype(float)
                sorted_idx = np.argsort(cat_scores)[::-1]
                count = 0
                for idx in sorted_idx:
                    if count >= per_cat_n: break
                    if cat_scores[idx] > 0 and idx not in seen:
                        result_idx.append(idx)
                        seen.add(idx)
                        count += 1

            # Isi sisa kuota dari global top (tanpa duplikat)
            global_top = np.argsort(scores)[::-1]
            for idx in global_top:
                if len(result_idx) >= top_n: break
                if scores[idx] > 0 and idx not in seen:
                    result_idx.append(idx)
                    seen.add(idx)

            result = self.df.iloc[result_idx].copy()
            result["cbf_score"] = scores[result_idx]

        else:
            # Single kategori: sort biasa
            top_idx = np.argsort(scores)[::-1][:top_n]
            result  = self.df.iloc[top_idx].copy()
            result["cbf_score"] = scores[top_idx]
            result = result[result["cbf_score"] > 0]

        return result.reset_index(drop=True)

    def interleave_by_category(self, df_candidates: pd.DataFrame,
                               categories: list, n: int) -> pd.DataFrame:
        """
        Urutkan kandidat secara selang-seling antar kategori.
        Contoh [Alam, Kuliner, Wisata]: A, K, W, A, K, W, ...
        Penting agar RL tidak cenderung pilih 1 kategori beruntun.
        """
        if len(categories) <= 1:
            return df_candidates.head(n)

        # Pisahkan per kategori, urutkan by cbf_score desc
        buckets = {cat: df_candidates[df_candidates["category"]==cat]
                       .sort_values("cbf_score", ascending=False)
                       .reset_index(drop=True)
                   for cat in categories}

        result = []
        pointers = {cat: 0 for cat in categories}
        seen_ids = set()
        # Round-robin
        while len(result) < n:
            added_this_round = 0
            for cat in categories:
                if len(result) >= n: break
                ptr  = pointers[cat]
                pool = buckets.get(cat, pd.DataFrame())
                while ptr < len(pool):
                    row = pool.iloc[ptr]
                    ptr += 1
                    if row["id"] not in seen_ids:
                        result.append(row)
                        seen_ids.add(row["id"])
                        added_this_round += 1
                        break
                pointers[cat] = ptr
            if added_this_round == 0:
                # Semua bucket habis, isi dari sisanya
                for _, row in df_candidates.iterrows():
                    if len(result) >= n: break
                    if row["id"] not in seen_ids:
                        result.append(row)
                        seen_ids.add(row["id"])
                break

        return pd.DataFrame(result).reset_index(drop=True) if result else df_candidates.head(n)


cbf_model = ContentBasedFilter(feature_matrix, df_clean, encoders)
cbf_model.fit()

# Simpan CBF
cbf_artifact = {
    "similarity_matrix": cbf_model.similarity_matrix,
    "feature_matrix":    feature_matrix,
    "df_index": df_clean[["id","name","category"]].to_dict("records"),
}
with open("models/cbf_model.pkl", "wb") as f:
    pickle.dump(cbf_artifact, f)
print("💾 models/cbf_model.pkl")

# Quick test — pastikan setiap kategori terwakili
test_cats = ["Alam", "Kuliner", "Wisata"]
rec = cbf_model.recommend(categories=test_cats, top_n=15)
print(f"\n🧪 Test multi-kategori {test_cats}:")
print(rec.groupby("category").size().to_dict())
assert all(c in rec["category"].values for c in test_cats), "❌ Ada kategori tidak terwakili!"
print("✅ Semua kategori terwakili dalam kandidat")


✅ CBF fitted. Similarity matrix: (1476, 1476)
💾 models/cbf_model.pkl

🧪 Test multi-kategori ['Alam', 'Kuliner', 'Wisata']:
{'Alam': 5, 'Kuliner': 5, 'Wisata': 5}
✅ Semua kategori terwakili dalam kandidat


## 📊 CELL 10 — Visualisasi CBF

In [15]:
# ============================================================
# CELL 9 — Training Content-Based Filtering
# ============================================================
def haversine_km(lat1, lng1, lat2, lng2) -> float:
    """Jarak Haversine (km) — modul global agar dipakai class lain."""
    R = 6371.0
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lng2 - lng1)
    a = math.sin(dphi / 2) ** 2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2) ** 2
    return 2 * R * math.asin(math.sqrt(a))


class ContentBasedFilter:
    """Cosine-similarity recommender atas vektor fitur destinasi."""

    def __init__(self, feature_matrix: np.ndarray, destinations_df: pd.DataFrame, encoders: dict):
        self.feature_matrix = feature_matrix
        self.df = destinations_df.reset_index(drop=True)
        self.encoders = encoders
        self.similarity_matrix = None

    def fit(self):
        self.similarity_matrix = cosine_similarity(self.feature_matrix)
        print(f"✅ CBF fitted. Similarity matrix shape: {self.similarity_matrix.shape}")
        return self

    # ---- helpers ----
    def _category_dims(self) -> int:
        return self.encoders.get("n_category_dims", 5)

    def _numeric_start(self) -> int:
        return self._category_dims()

    def _numeric_end(self) -> int:
        return self._numeric_start() + self.encoders.get("n_numeric_dims", 5)

    def build_user_profile(self, categories: list, budget: float = None) -> np.ndarray:
        """Rata-rata fitur destinasi pada kategori user; budget di-override ke ticket dim."""
        if not categories:
            mask = np.ones(len(self.df), dtype=bool)
        else:
            mask = self.df["category"].isin(categories).values
        if mask.sum() == 0:
            mask = np.ones(len(self.df), dtype=bool)
        profile = self.feature_matrix[mask].mean(axis=0)

        # Override ticket_norm bila budget given.
        if budget is not None and budget > 0:
            scaler = self.encoders["scaler"]
            num_cols = self.encoders["feature_cols"]
            ticket_idx_in_numeric = num_cols.index("ticket_log")
            # Bangun row dummy dengan median per fitur supaya scaler stabil
            dummy = np.zeros((1, len(num_cols)))
            for j, col in enumerate(num_cols):
                if col == "ticket_log":
                    dummy[0, j] = math.log1p(budget)
                else:
                    dummy[0, j] = self.df[col if col != "ticket_log" else "ticket"].median() \
                        if col != "ticket_log" else math.log1p(self.df["ticket"].median())
            scaled = scaler.transform(dummy)[0]
            profile_idx = self._numeric_start() + ticket_idx_in_numeric
            profile[profile_idx] = scaled[ticket_idx_in_numeric]
        return profile.reshape(1, -1)

    def recommend(
        self,
        categories: list = None,
        budget: float = None,
        max_km: float = None,
        home_lat: float = None,
        home_lng: float = None,
        top_n: int = 20,
    ) -> pd.DataFrame:
        if self.similarity_matrix is None:
            self.fit()
        categories = categories or []
        user_profile = self.build_user_profile(categories, budget=budget)
        sims = cosine_similarity(user_profile, self.feature_matrix).flatten()

        out = self.df.copy()
        out["cbf_score"] = sims

        # Hard constraint: kategori
        if categories:
            mask = out["category"].isin(categories)
            if mask.sum() > 0:
                out = out[mask]
        # Budget: ticket <= budget
        if budget is not None:
            out_b = out[out["ticket"].astype(float) <= float(budget)]
            if len(out_b) > 0:
                out = out_b
        # Distance: haversine dari home <= max_km
        if max_km is not None and home_lat is not None and home_lng is not None:
            out = out.copy()
            out["dist_home_km"] = out.apply(
                lambda r: haversine_km(home_lat, home_lng, r["lat"], r["lng"]),
                axis=1,
            )
            within = out[out["dist_home_km"] <= float(max_km)]
            if len(within) > 0:
                out = within
        return out.sort_values("cbf_score", ascending=False).head(top_n).reset_index(drop=True)

    def get_similar_destinations(self, dest_id: str, top_n: int = 5) -> pd.DataFrame:
        if self.similarity_matrix is None:
            self.fit()
        idx_list = self.df.index[self.df["id"] == dest_id].tolist()
        if not idx_list:
            return self.df.head(0)
        idx = idx_list[0]
        sims = self.similarity_matrix[idx]
        order = np.argsort(-sims)
        order = [i for i in order if i != idx][:top_n]
        out = self.df.iloc[order].copy()
        out["sim_score"] = sims[order]
        return out.reset_index(drop=True)

    def save(self, path: str):
        with open(path, "wb") as f:
            pickle.dump({
                "similarity_matrix": self.similarity_matrix,
                "feature_matrix": self.feature_matrix,
                "df_index": self.df[["id", "name", "category"]].to_dict("records"),
            }, f)
        print(f"✅ CBF model disimpan ke {path}")

    @classmethod
    def load(cls, path: str, feature_matrix, destinations_df, encoders):
        with open(path, "rb") as f:
            data = pickle.load(f)
        obj = cls(feature_matrix, destinations_df, encoders)
        obj.similarity_matrix = data["similarity_matrix"]
        return obj


cbf_model = ContentBasedFilter(feature_matrix, df_clean, encoders).fit()

# ---- 3 quick tests ----
print("\n🧪 TEST 1 — Alam, budget 200k, top 5")
t1 = cbf_model.recommend(categories=["Alam"], budget=200000, top_n=5)
print(t1[["name", "category", "ticket", "rating", "cbf_score"]])

print("\n🧪 TEST 2 — Kuliner+Budaya, no budget, top 10")
t2 = cbf_model.recommend(categories=["Kuliner", "Budaya"], top_n=10)
print(t2[["name", "category", "ticket", "rating", "cbf_score"]])

print("\n🧪 TEST 3 — Wisata+Belanja, budget 500k, max 30km dari Alun-Alun, top 8")
t3 = cbf_model.recommend(
    categories=["Wisata", "Belanja"], budget=500000, max_km=30,
    home_lat=-6.9215, home_lng=107.6071, top_n=8,
)
cols = ["name", "category", "ticket", "rating", "cbf_score"]
if "dist_home_km" in t3.columns:
    cols.append("dist_home_km")
print(t3[cols])

# Simpan model
cbf_model.save("models/cbf_model.pkl")


✅ CBF fitted. Similarity matrix shape: (1476, 1476)

🧪 TEST 1 — Alam, budget 200k, top 5
                   name category  ticket  rating  cbf_score
0        Tebing Keraton     Alam   15000     4.5   0.981569
1    Cagar Alam Junghun     Alam   20000     4.5   0.981569
2            Bukit Moko     Alam   20000     4.5   0.980308
3  FCL front lobby park     Alam   20000     4.5   0.978362
4      Tangkuban Perahu     Alam   30000     4.5   0.978342

🧪 TEST 2 — Kuliner+Budaya, no budget, top 10
                              name category  ticket  rating  cbf_score
0                     RM Alas Daun  Kuliner       0     4.3   0.995659
1  Rumah Makan Taliwang Bersaudara  Kuliner       0     4.3   0.995657
2                     Roemah Nenek  Kuliner       0     4.3   0.995653
3                     Simpang Raya  Kuliner       0     4.3   0.995651
4                    Bale Gazeeboe  Kuliner       0     4.3   0.995651
5                Glosis Restaurant  Kuliner       0     4.3   0.995650
6       

# ============================================================
# CELL 10 — Visualisasi CBF
# ============================================================
sns.set_theme(style="whitegrid")

# (a) Heatmap similarity matrix — subsample 20 destinasi
sample_n = min(20, len(df_clean))
sub_idx = np.linspace(0, len(df_clean) - 1, sample_n, dtype=int)
sub_sim = cbf_model.similarity_matrix[np.ix_(sub_idx, sub_idx)]
sub_names = df_clean.iloc[sub_idx]["name"].tolist()

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(sub_sim, xticklabels=sub_names, yticklabels=sub_names,
            cmap="viridis", ax=ax, cbar_kws={"label": "Cosine Similarity"})
ax.set_title("Cosine Similarity Matrix — Top 20 Destinations", fontsize=13)
plt.xticks(rotation=70, ha="right", fontsize=8)
plt.yticks(fontsize=8)
plt.tight_layout()
plt.savefig("data/processed/cbf_similarity_heatmap.png", dpi=120)
plt.show()

# (b) Bar chart: rata-rata cbf_score per kategori
print("\nMenghitung average CBF score per kategori (sample 100 random user profiles)…")
score_per_cat = defaultdict(list)
for _ in range(100):
    cats = random.sample(["Alam", "Kuliner", "Budaya", "Wisata", "Belanja"],
                        k=random.randint(1, 3))
    rec = cbf_model.recommend(categories=cats, top_n=10)
    if len(rec):
        for cat, sub in rec.groupby("category"):
            score_per_cat[cat].extend(sub["cbf_score"].tolist())

avg_scores = {k: float(np.mean(v)) for k, v in score_per_cat.items() if v}
fig, ax = plt.subplots(figsize=(8, 5))
cats_sorted = sorted(avg_scores, key=avg_scores.get)
ax.barh(cats_sorted, [avg_scores[c] for c in cats_sorted], color="steelblue")
ax.set_xlabel("Average CBF Score (cosine similarity)")
ax.set_title("Average CBF Score per Kategori (100 random user profiles)")
plt.tight_layout()
plt.savefig("data/processed/cbf_category_scores.png", dpi=120)
plt.show()

print("✅ data/processed/cbf_similarity_heatmap.png")
print("✅ data/processed/cbf_category_scores.png")


## 🤖 CELL 12 — Training RL Agent (Variety-Aware)

> **Perubahan reward function:**
> - `variety_bonus`: 0.2 → **0.4** (lebih besar agar RL aktif pilih kategori baru)
> - Tambah `consecutive_penalty`: -0.3 jika pilih kategori sama 2× berturut-turut
> - Kandidat di-interleave dari awal (round-robin per kategori)


In [17]:
# ============================================================
# FIX — Tambahkan interleave_by_category ke cbf_model
# Jalankan cell ini SEBELUM Cell 12 (Training RL)
# ============================================================
import types

def interleave_by_category(self, df_candidates, categories, n):
    """
    Urutkan kandidat secara round-robin antar kategori.
    Contoh [Alam, Kuliner, Wisata]: A, K, W, A, K, W, ...
    """
    if len(categories) <= 1:
        return df_candidates.head(n)

    buckets  = {
        cat: df_candidates[df_candidates["category"] == cat]
                 .sort_values("cbf_score", ascending=False)
                 .reset_index(drop=True)
        for cat in categories
    }
    result   = []
    pointers = {cat: 0 for cat in categories}
    seen_ids = set()

    while len(result) < n:
        added_this_round = 0
        for cat in categories:
            if len(result) >= n:
                break
            ptr  = pointers[cat]
            pool = buckets.get(cat, pd.DataFrame())
            while ptr < len(pool):
                row = pool.iloc[ptr]
                ptr += 1
                if row["id"] not in seen_ids:
                    result.append(row)
                    seen_ids.add(row["id"])
                    added_this_round += 1
                    break
            pointers[cat] = ptr

        if added_this_round == 0:
            # Semua bucket habis, isi sisa dari df_candidates
            for _, row in df_candidates.iterrows():
                if len(result) >= n:
                    break
                if row["id"] not in seen_ids:
                    result.append(row)
                    seen_ids.add(row["id"])
            break

    return pd.DataFrame(result).reset_index(drop=True) if result else df_candidates.head(n)

# Bind method ke instance cbf_model yang sudah ada
cbf_model.interleave_by_category = types.MethodType(interleave_by_category, cbf_model)

# Verifikasi
test = cbf_model.recommend(categories=["Alam","Kuliner"], top_n=10)
result = cbf_model.interleave_by_category(test, ["Alam","Kuliner"], n=10)
print(f"✅ interleave_by_category berhasil ditambahkan")
print(f"   Urutan kategori: {result['category'].tolist()}")

✅ interleave_by_category berhasil ditambahkan
   Urutan kategori: ['Alam', 'Alam', 'Alam', 'Alam', 'Alam', 'Alam', 'Alam', 'Alam', 'Alam', 'Alam']


## 📊 CELL 13 — Visualisasi Training RL

In [18]:
# ============================================================
# CELL 12 — Training Q-Learning RL Agent
# ============================================================
HOME_OPTIONS = [
    {"lat": -6.9215, "lng": 107.6071, "name": "Alun-Alun Bandung"},
    {"lat": -6.9145, "lng": 107.6020, "name": "Stasiun Bandung"},
    {"lat": -6.8126, "lng": 107.6178, "name": "Pasar Lembang"},
    {"lat": -6.8915, "lng": 107.6107, "name": "Dago"},
    {"lat": -6.9024, "lng": 107.6188, "name": "Gedung Sate"},
]
SPEED_KMH = 28


class BandungTravelEnv:
    """Simulated env: user memilih destinasi satu per satu (sequential)."""

    REWARD_WEIGHTS = {"rating": 0.5, "variety": 0.2, "budget_eff": 0.3}
    PENALTY_OVERTIME = 1.0
    PENALTY_OVERBUDGET = 0.5

    def __init__(self, destinations_df: pd.DataFrame, cbf_model: ContentBasedFilter):
        self.df = destinations_df.reset_index(drop=True)
        self.cbf = cbf_model
        self.n_destinations = len(self.df)
        self.params = None
        self.candidates = []  # indeks pada self.df
        self.selected = []    # indeks pada self.df
        self.spent = 0
        self.cur_lat = None
        self.cur_lng = None
        self.cur_time = 0

    # ---- helpers ----
    def _haversine(self, lat1, lng1, lat2, lng2):
        return haversine_km(lat1, lng1, lat2, lng2)

    def _idx_by_id(self, dest_id: str) -> int:
        rows = self.df.index[self.df["id"] == dest_id].tolist()
        return rows[0] if rows else -1

    # ---- API ----
    def reset(self, params: dict):
        self.params = {**params}
        self.params.setdefault("budget", None)
        self.params.setdefault("max_km", None)
        if self.params["budget"] is None:
            self.params["budget"] = 999_999_999
        if self.params["max_km"] is None:
            self.params["max_km"] = 999
        self.params.setdefault("count", 4)
        self.params.setdefault("startMin", 9 * 60)
        self.params.setdefault("endMin", 21 * 60)
        self.params.setdefault("home_lat", HOME_OPTIONS[0]["lat"])
        self.params.setdefault("home_lng", HOME_OPTIONS[0]["lng"])

        # CBF top-30 jadi kandidat awal
        rec = self.cbf.recommend(
            categories=self.params.get("categories", []),
            budget=None if self.params["budget"] >= 999_999_998 else self.params["budget"],
            max_km=None if self.params["max_km"] >= 999 else self.params["max_km"],
            home_lat=self.params["home_lat"],
            home_lng=self.params["home_lng"],
            top_n=30,
        )
        if len(rec) == 0:
            rec = self.cbf.recommend(top_n=30)
        self.candidates = [self._idx_by_id(i) for i in rec["id"].tolist()]
        self.candidates = [i for i in self.candidates if i >= 0]

        self.selected = []
        self.spent = 0
        self.cur_lat = self.params["home_lat"]
        self.cur_lng = self.params["home_lng"]
        self.cur_time = self.params["startMin"]

        return self._get_state(), list(self.candidates)

    def _get_state(self):
        n_sel = min(8, len(self.selected))
        budget_total = self.params["budget"]
        # Bucket sesuai spec README:
        # 0 = habis, 1 = <25%, 2 = <50%, 3 = <75%, 4 = >=75% sisa
        if budget_total <= 0:
            budget_level = 0
        elif budget_total >= 999_999_998:
            # Effectively no budget limit
            budget_level = 4
        else:
            ratio = max(0.0, 1.0 - self.spent / budget_total)
            if ratio <= 0.0:
                budget_level = 0
            elif ratio < 0.25:
                budget_level = 1
            elif ratio < 0.50:
                budget_level = 2
            elif ratio < 0.75:
                budget_level = 3
            else:
                budget_level = 4
        time_left = max(0, self.params["endMin"] - self.cur_time)
        if time_left <= 0:
            time_level = 0
        elif time_left < 120:
            time_level = 1
        elif time_left < 240:
            time_level = 2
        elif time_left < 360:
            time_level = 3
        else:
            time_level = 4
        dom = 0
        if self.selected:
            cats = Counter(self.df.iloc[i]["category"] for i in self.selected)
            top_cat = cats.most_common(1)[0][0]
            order = ["Alam", "Kuliner", "Budaya", "Wisata", "Belanja"]
            dom = order.index(top_cat) if top_cat in order else 0
        return (n_sel, budget_level, time_level, dom)

    def get_valid_actions(self):
        valid = []
        for k, idx in enumerate(self.candidates):
            if idx in self.selected:
                continue
            row = self.df.iloc[idx]
            if int(row["ticket"]) > (self.params["budget"] - self.spent):
                continue
            dist_home = self._haversine(
                self.params["home_lat"], self.params["home_lng"], row["lat"], row["lng"]
            )
            if dist_home > self.params["max_km"]:
                continue
            travel_km = self._haversine(self.cur_lat, self.cur_lng, row["lat"], row["lng"])
            travel_min = (travel_km / SPEED_KMH) * 60
            arrive = self.cur_time + travel_min
            depart = arrive + int(row["duration"])
            return_min = (self._haversine(
                row["lat"], row["lng"],
                self.params["home_lat"], self.params["home_lng"]
            ) / SPEED_KMH) * 60
            if depart + return_min > self.params["endMin"]:
                continue
            valid.append(k)
        return valid

    def _calculate_reward(self, dest_row, travel_km: float) -> float:
        rating_score = float(dest_row["rating"]) / 5.0
        # NOTE: dest_row sudah masuk ke self.selected & self.spent saat method ini dipanggil
        # (lihat step()), jadi cek variety harus exclude destinasi terakhir,
        # dan budget_eff tidak boleh mengurangi ticket lagi (sudah termasuk di self.spent).
        prior_selected = self.selected[:-1] if self.selected else []
        cats_chosen = {self.df.iloc[i]["category"] for i in prior_selected}
        variety = 0.2 if dest_row["category"] not in cats_chosen else 0.0

        budget_total = self.params["budget"]
        if budget_total <= 0 or budget_total >= 999_999_998:
            budget_eff = 0.5
        else:
            remain = max(0, budget_total - self.spent)  # spent already includes current ticket
            budget_eff = remain / budget_total

        w = self.REWARD_WEIGHTS
        r = w["rating"] * rating_score + w["variety"] * variety + w["budget_eff"] * budget_eff

        # Penalti
        if self.cur_time > self.params["endMin"]:
            r -= self.PENALTY_OVERTIME
        if self.spent > self.params["budget"]:
            r -= self.PENALTY_OVERBUDGET
        return float(r)

    def step(self, action_idx: int):
        valid = self.get_valid_actions()
        if action_idx not in valid:
            # Tidak valid → episode done dengan penalti kecil
            return self._get_state(), -0.1, True, {"reason": "invalid_action"}

        idx = self.candidates[action_idx]
        row = self.df.iloc[idx]
        travel_km = self._haversine(self.cur_lat, self.cur_lng, row["lat"], row["lng"])
        travel_min = (travel_km / SPEED_KMH) * 60
        self.cur_time += travel_min
        self.cur_time += int(row["duration"])
        self.spent += int(row["ticket"])
        self.selected.append(idx)
        self.cur_lat, self.cur_lng = row["lat"], row["lng"]

        reward = self._calculate_reward(row, travel_km)
        done = (
            len(self.selected) >= self.params["count"]
            or self.cur_time >= self.params["endMin"]
            or len(self.get_valid_actions()) == 0
        )
        return self._get_state(), reward, done, {"travel_km": travel_km}

    def generate_random_params(self) -> dict:
        cats_all = ["Alam", "Kuliner", "Budaya", "Wisata", "Belanja"]
        n_cats = random.randint(1, len(cats_all))
        cats = random.sample(cats_all, k=n_cats)
        budget = None if random.random() < 0.20 else random.randint(50_000, 2_000_000)
        max_km = None if random.random() < 0.40 else random.randint(20, 80)
        count = random.randint(2, 6)
        start = random.randint(420, 600)
        end = start + random.randint(300, 900)
        home = random.choice(HOME_OPTIONS)
        return {
            "categories": cats,
            "budget": budget,
            "max_km": max_km,
            "count": count,
            "startMin": start,
            "endMin": end,
            "home_lat": home["lat"],
            "home_lng": home["lng"],
        }


class QLearningAgent:
    """Tabular Q-Learning dengan epsilon-greedy."""

    def __init__(self, learning_rate=0.1, discount_factor=0.95,
                 epsilon=1.0, epsilon_min=0.05, epsilon_decay=0.995):
        self.lr = learning_rate
        self.gamma = discount_factor
        self.epsilon = epsilon
        self.epsilon_min = epsilon_min
        self.epsilon_decay = epsilon_decay
        self.q_table = defaultdict(lambda: defaultdict(float))
        self.training_history = []

    @staticmethod
    def _action_key(action_idx: int, candidate_ids: list) -> str:
        return candidate_ids[action_idx] if 0 <= action_idx < len(candidate_ids) else "unknown"

    def choose_action(self, state, valid_actions, candidate_ids):
        if not valid_actions:
            return -1
        if random.random() < self.epsilon:
            return random.choice(valid_actions)
        best, best_q = valid_actions[0], -float("inf")
        for a in valid_actions:
            q = self.q_table[state][self._action_key(a, candidate_ids)]
            if q > best_q:
                best_q, best = q, a
        return best

    def update(self, state, action_idx, reward, next_state, done,
               valid_next_actions, candidate_ids, next_candidate_ids):
        a_key = self._action_key(action_idx, candidate_ids)
        cur_q = self.q_table[state][a_key]
        if done or not valid_next_actions:
            target = reward
        else:
            future = max(
                self.q_table[next_state][self._action_key(a, next_candidate_ids)]
                for a in valid_next_actions
            )
            target = reward + self.gamma * future
        self.q_table[state][a_key] = cur_q + self.lr * (target - cur_q)

    def decay_epsilon(self):
        self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)

    def save(self, path: str):
        plain = {s: dict(actions) for s, actions in self.q_table.items()}
        data = {
            "q_table": plain,
            "epsilon": self.epsilon,
            "lr": self.lr,
            "gamma": self.gamma,
            "training_history": self.training_history,
        }
        with open(path, "wb") as f:
            pickle.dump(data, f)
        print(f"✅ RL Agent tersimpan ke {path}")

    @classmethod
    def load(cls, path: str):
        with open(path, "rb") as f:
            data = pickle.load(f)
        ag = cls(learning_rate=data["lr"], discount_factor=data["gamma"])
        ag.q_table = defaultdict(lambda: defaultdict(float))
        for s, actions in data["q_table"].items():
            for a, v in actions.items():
                ag.q_table[s][a] = v
        ag.epsilon = data["epsilon"]
        ag.training_history = data.get("training_history", [])
        return ag


def train_rl_agent(env: BandungTravelEnv, agent: QLearningAgent,
                   n_episodes: int = 3000, log_interval: int = 500) -> list:
    rewards_log = []
    for ep in tqdm(range(n_episodes), desc="Training RL Agent"):
        params = env.generate_random_params()
        state, candidates = env.reset(params)
        candidate_ids = [env.df.iloc[i]["id"] for i in candidates]
        total = 0.0
        done = False
        while not done:
            valid = env.get_valid_actions()
            if not valid:
                break
            action = agent.choose_action(state, valid, candidate_ids)
            if action < 0:
                break
            next_state, reward, done, _ = env.step(action)
            next_valid = env.get_valid_actions()
            agent.update(state, action, reward, next_state, done,
                         next_valid, candidate_ids, candidate_ids)
            state = next_state
            total += reward
        agent.decay_epsilon()
        rewards_log.append(total)
        agent.training_history.append(total)
        if (ep + 1) % log_interval == 0:
            avg = float(np.mean(rewards_log[-log_interval:]))
            print(f"  Episode {ep+1}/{n_episodes} | avg reward {avg:.4f} | ε {agent.epsilon:.3f}")
    return rewards_log


env = BandungTravelEnv(df_clean, cbf_model)
rl_agent = QLearningAgent()
N_EPISODES = 3000
rewards_log = train_rl_agent(env, rl_agent, n_episodes=N_EPISODES, log_interval=500)
rl_agent.save("models/rl_agent.pkl")


Training RL Agent:  17%|█▋        | 502/3000 [00:18<01:31, 27.41it/s]

  Episode 500/3000 | avg reward 2.2343 | ε 0.082


Training RL Agent:  33%|███▎      | 1004/3000 [00:36<01:06, 30.21it/s]

  Episode 1000/3000 | avg reward 2.2399 | ε 0.050


Training RL Agent:  50%|█████     | 1505/3000 [00:54<00:51, 29.31it/s]

...

  Episode 2000/3000 | avg reward 2.1779 | ε 0.050


Training RL Agent:  84%|████████▎ | 2505/3000 [01:30<00:19, 25.62it/s]

  Episode 2500/3000 | avg reward 2.3150 | ε 0.050


Training RL Agent: 100%|██████████| 3000/3000 [01:48<00:00, 27.55it/s]

  Episode 3000/3000 | avg reward 2.3114 | ε 0.050
✅ RL Agent tersimpan ke models/rl_agent.pkl


# ============================================================
# CELL 13 — Visualisasi Training RL
# ============================================================
rewards_arr = np.array(rewards_log)
window = 100
rolling = pd.Series(rewards_arr).rolling(window=window, min_periods=1).mean().values

# (a) Learning curve
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(rewards_arr, alpha=0.25, label="raw reward")
ax.plot(rolling, color="C1", linewidth=2, label=f"rolling mean (w={window})")
ax.set_xlabel("Episode"); ax.set_ylabel("Total Reward")
ax.set_title(f"RL Agent Learning Curve — {len(rewards_arr)} Episodes")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("data/processed/rl_learning_curve.png", dpi=120)
plt.show()

# (b) Epsilon decay (rekonstruksi)
ep_curve = []
eps = 1.0
for _ in range(len(rewards_arr)):
    ep_curve.append(eps)
    eps = max(rl_agent.epsilon_min, eps * rl_agent.epsilon_decay)
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(ep_curve, color="purple")
ax.set_xlabel("Episode"); ax.set_ylabel("Epsilon")
ax.set_title("Epsilon Decay Schedule")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("data/processed/rl_epsilon_decay.png", dpi=120)
plt.show()

# (c) Reward distribution awal vs akhir
first_chunk = rewards_arr[:500] if len(rewards_arr) >= 500 else rewards_arr[:len(rewards_arr)//2]
last_chunk = rewards_arr[-500:] if len(rewards_arr) >= 500 else rewards_arr[len(rewards_arr)//2:]
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(first_chunk, bins=30, alpha=0.6, label="first 500 episodes", color="tomato")
ax.hist(last_chunk, bins=30, alpha=0.6, label="last 500 episodes", color="seagreen")
ax.set_xlabel("Total Reward"); ax.set_ylabel("Frequency")
ax.set_title("Reward Distribution: First vs Last 500 Episodes")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("data/processed/rl_reward_distribution.png", dpi=120)
plt.show()

print("✅ Visualisasi tersimpan: rl_learning_curve.png, rl_epsilon_decay.png, rl_reward_distribution.png")


## 🗺️ Fase 3C: Route Optimizer — TSP Nearest-Neighbor

Setelah RL Agent memilih N destinasi, urutan kunjungan dioptimalkan:
1. **Haversine Distance** untuk fallback offline
2. **OSRM API** (`router.project-osrm.org`) untuk waktu tempuh nyata bila tersedia
3. **Nearest-Neighbor Heuristic** untuk route ordering

OSRM endpoint:
```
GET http://router.project-osrm.org/route/v1/driving/{lng1},{lat1};{lng2},{lat2}?overview=false
```
Response → `routes[0].distance` (meter), `routes[0].duration` (detik).


## 🔗 CELL 16 — Inference: Full Pipeline + Category Guarantee

> **Perubahan kritis:**
> - `enforce_category_guarantee()`: setelah RL memilih, cek apakah semua kategori terwakili
> - Jika ada yang hilang → swap destinasi dari kategori over-represented
> - Test suite: validasi variety (tidak ada 3+ berturut-turut kategori sama)


In [24]:
# ============================================================
# CELL 15 — RouteOptimizer Class
# ============================================================
class RouteOptimizer:
    SPEED_KMH = 28
    OSRM_BASE = "http://router.project-osrm.org/route/v1/driving"

    def __init__(self, use_osrm=True, osrm_timeout=5.0):
        self.use_osrm     = use_osrm
        self.osrm_timeout = osrm_timeout
        self._osrm_cache  = {}

    def haversine_km(self, lat1, lng1, lat2, lng2):
        return haversine_km(lat1, lng1, lat2, lng2)

    def osrm_travel_time(self, lat1, lng1, lat2, lng2):
        if not self.use_osrm:
            d = self.haversine_km(lat1, lng1, lat2, lng2)
            return d, (d / self.SPEED_KMH) * 60
        key = (round(lat1,4), round(lng1,4), round(lat2,4), round(lng2,4))
        if key in self._osrm_cache:
            return self._osrm_cache[key]
        try:
            url  = f"{self.OSRM_BASE}/{lng1},{lat1};{lng2},{lat2}?overview=false"
            resp = requests.get(url, timeout=self.osrm_timeout)
            resp.raise_for_status()
            route   = resp.json()["routes"][0]
            dist_km = route["distance"] / 1000.0
            dur_min = route["duration"] / 60.0
        except Exception:
            d = self.haversine_km(lat1, lng1, lat2, lng2)
            dist_km, dur_min = d, (d / self.SPEED_KMH) * 60
        self._osrm_cache[key] = (dist_km, dur_min)
        return dist_km, dur_min

    def nearest_neighbor_route(self, home: dict, destinations: list) -> list:
        if not destinations:
            return []
        remaining = list(destinations)
        ordered   = []
        cur_lat, cur_lng = home["lat"], home["lng"]
        while remaining:
            nxt = min(remaining,
                      key=lambda d: self.haversine_km(cur_lat, cur_lng, d["lat"], d["lng"]))
            ordered.append(nxt)
            remaining = [d for d in remaining if d.get("id") != nxt.get("id")]
            cur_lat, cur_lng = nxt["lat"], nxt["lng"]
        return ordered

    def build_itinerary(self, home: dict, home_name: str,
                        ordered_destinations: list,
                        start_min: int, end_min: int) -> dict:
        steps      = []
        cur_time   = start_min
        cur_lat, cur_lng = home["lat"], home["lng"]
        total_cost = 0
        total_km   = 0.0

        for idx, dest in enumerate(ordered_destinations, 1):
            dist_km, dur_min = self.osrm_travel_time(
                cur_lat, cur_lng, dest["lat"], dest["lng"]
            )
            travel_min = max(1, int(dur_min))
            arrive_at  = cur_time + travel_min
            stay_min   = int(dest.get("duration", 90))
            depart_at  = arrive_at + stay_min

            steps.append({
                "idx":  idx,
                "dest": {
                    "id":         dest.get("id", ""),
                    "name":       dest.get("name", ""),
                    "category":   dest.get("category", ""),
                    "desc":       dest.get("desc", ""),
                    "ticket":     int(dest.get("ticket", 0)),
                    "duration":   stay_min,
                    "lat":        float(dest["lat"]),
                    "lng":        float(dest["lng"]),
                    "rating":     float(dest.get("rating", 0)),
                    "tags":       dest.get("tags", []),
                    "gmaps_url":  dest.get("gmaps_url", ""),
                    "stay_detail": dest.get("stay_detail", ""),
                },
                "travelMin": travel_min,
                "travelKm":  round(dist_km, 2),
                "arriveAt":  arrive_at,
                "departAt":  depart_at,
            })

            total_cost += int(dest.get("ticket", 0))
            total_km   += dist_km
            cur_time    = depart_at
            cur_lat, cur_lng = dest["lat"], dest["lng"]

        # Perjalanan pulang ke home
        ret_km, ret_min = self.osrm_travel_time(
            cur_lat, cur_lng, home["lat"], home["lng"]
        )
        ret_min    = max(1, int(ret_min))
        arrive_home = cur_time + ret_min

        return {
            "steps":      steps,
            "totalCost":  total_cost,
            "totalKm":    round(total_km + ret_km, 2),
            "totalTime":  arrive_home - start_min,
            "returnKm":   round(ret_km, 2),
            "returnMin":  ret_min,
            "arriveHome": arrive_home,
            "overBudget": False,
            "spareMin":   max(0, end_min - arrive_home),
        }


route_optimizer = RouteOptimizer()
print("✅ RouteOptimizer siap")

✅ RouteOptimizer siap


In [25]:
# ============================================================
# FIX — Instansiasi route_optimizer
# Jalankan SEBELUM Cell 16
# ============================================================
route_optimizer = RouteOptimizer()
print("✅ route_optimizer siap")

✅ route_optimizer siap


In [26]:
# ============================================================
# CELL 16 — Inference: Full Pipeline + Category Guarantee
# ============================================================
# Perubahan v2:
# - rl_select_destinations() enforce: setiap kategori yang diminta
#   user WAJIB ada minimal 1 di hasil akhir
# - Post-process: jika ada kategori yang tidak terwakili setelah
#   RL memilih, paksa swap 1 destinasi dengan destinasi dari
#   kategori yang hilang (dengan CBF score tertinggi)

def rl_select_destinations(env, agent, params: dict) -> list:
    """Inference RL greedy (epsilon=0)."""
    saved_eps    = agent.epsilon
    agent.epsilon = 0.0
    try:
        state, candidates = env.reset(params)
        candidate_ids     = [env.df.iloc[i]["id"] for i in env.candidates]
        target = max(1, int(params.get("count", 4)))
        done   = False
        while not done and len(env.selected) < target:
            valid = env.get_valid_actions()
            if not valid: break
            action = agent.choose_action(state, valid, candidate_ids)
            if action < 0 or action >= len(env.candidates): break
            state, _, done, _ = env.step(action)
        return [env.df.iloc[i].to_dict() for i in env.selected]
    finally:
        agent.epsilon = saved_eps


def enforce_category_guarantee(selected: list, params: dict) -> list:
    """
    Pastikan setiap kategori yang diminta user terwakili minimal 1.
    Jika ada yang tidak terwakili:
    1. Ambil kandidat CBF terbaik dari kategori yang hilang
    2. Swap dengan destinasi yang paling banyak mewakili kategori
       yang sudah over-represented (tanpa melanggar budget)
    """
    categories = params.get("categories", [])
    if len(categories) <= 1:
        return selected  # single kategori, tidak perlu guarantee

    missing_cats = [c for c in categories
                    if c not in {d["category"] for d in selected}]
    if not missing_cats:
        return selected  # semua sudah terwakili

    budget   = params.get("budget")
    has_budget = budget is not None and budget < 999_999_998
    home_lat = params["home"]["lat"]
    home_lng = params["home"]["lng"]

    for missing_cat in missing_cats:
        # Cari kandidat dari kategori yang hilang
        rec = cbf_model.recommend(
            categories=[missing_cat], budget=budget,
            max_km=params.get("maxKm"),
            home_lat=home_lat, home_lng=home_lng, top_n=10
        )
        existing_ids = {d["id"] for d in selected}
        candidate = None
        for _, row in rec.iterrows():
            if row["id"] not in existing_ids:
                if has_budget:
                    # Hitung sisa budget setelah swap
                    total_now = sum(int(d.get("ticket",0)) for d in selected)
                    if int(row.get("ticket",0)) <= budget:
                        candidate = row.to_dict()
                        break
                else:
                    candidate = row.to_dict()
                    break

        if not candidate:
            continue  # tidak bisa paksa, skip

        # Temukan kategori yang paling over-represented untuk diswap
        cat_counts = Counter(d["category"] for d in selected)
        # Pilih kategori dengan count terbanyak (bukan yang missing)
        over_cat   = max(
            [c for c in cat_counts if c != missing_cat and cat_counts[c] > 1],
            key=lambda c: cat_counts[c],
            default=None
        )
        if over_cat is None:
            # Tidak ada yang over-rep, swap destinasi non-prioritas
            # (destinasi dengan rating terendah dari bukan missing_cat)
            swap_target = min(
                [d for d in selected if d["category"] != missing_cat],
                key=lambda d: d.get("rating", 0),
                default=None
            )
        else:
            # Swap destinasi rating terendah dari kategori over-rep
            swap_target = min(
                [d for d in selected if d["category"] == over_cat],
                key=lambda d: d.get("rating", 0),
                default=None
            )

        if swap_target:
            idx = next(i for i, d in enumerate(selected) if d["id"] == swap_target["id"])
            selected[idx] = candidate
            print(f"  🔄 Swap: '{swap_target['name']}' ({swap_target['category']}) "
                  f"→ '{candidate['name']}' ({missing_cat})")

    return selected


def _smart_fallback_fill(selected, params, target):
    """Lengkapi jika kurang dari target."""
    if len(selected) >= target: return selected
    home_lat   = params["home"]["lat"]
    home_lng   = params["home"]["lng"]
    has_budget = params.get("budget") is not None
    spent      = sum(int(d.get("ticket",0)) for d in selected)
    remaining  = (params["budget"] - spent) if has_budget else None

    rec  = cbf_model.recommend(
        categories=params.get("categories",[]), budget=remaining,
        max_km=params.get("maxKm"), home_lat=home_lat, home_lng=home_lng, top_n=30
    )
    seen = {d["id"] for d in selected}
    for _, r in rec.iterrows():
        if len(selected) >= target: break
        if r["id"] in seen: continue
        if has_budget and int(r.get("ticket",0)) > remaining: continue
        selected.append(r.to_dict())
        seen.add(r["id"])
        if has_budget: remaining -= int(r.get("ticket",0))
    return selected


def full_pipeline_test(params: dict) -> dict:
    """Full pipeline: CBF → RL → Guarantee → Fallback → TSP → itinerary."""
    target = max(1, int(params.get("count", 4)))
    rl_params = {
        "categories": params.get("categories",[]),
        "budget":     params.get("budget"),
        "max_km":     params.get("maxKm"),
        "count":      target,
        "startMin":   params.get("startMin", 9*60),
        "endMin":     params.get("endMin", 21*60),
        "home_lat":   params["home"]["lat"],
        "home_lng":   params["home"]["lng"],
    }

    # RL selection
    selected = rl_select_destinations(env, rl_agent, rl_params)

    # Category guarantee (kunci perubahan v2)
    selected = enforce_category_guarantee(selected, params)

    # Fallback jika kurang
    selected = _smart_fallback_fill(selected, params, target)

    if not selected:
        return {"steps":[],"totalCost":0,"totalKm":0,"totalTime":0,
                "returnKm":0,"returnMin":0,"arriveHome":params.get("startMin",540),
                "overBudget":False,"spareMin":0}

    ordered  = route_optimizer.nearest_neighbor_route(params["home"], selected)
    result   = route_optimizer.build_itinerary(
        home=params["home"], home_name=params.get("homeName","Home"),
        ordered_destinations=ordered,
        start_min=params.get("startMin",9*60), end_min=params.get("endMin",21*60)
    )
    budget = params.get("budget")
    result["overBudget"] = bool(budget and budget < 999_999_998 and result["totalCost"] > budget)
    return result


# ── Test suite ────────────────────────────────────────────────
print("🧪 Integration Test\n")

test_cases = [
    {"desc": "Multi-kategori 3 (Alam+Kuliner+Wisata)",
     "params": {"home":{"lat":-6.9215,"lng":107.6071},"homeName":"Alun-Alun",
                "count":4,"maxKm":None,"startMin":9*60,"endMin":20*60,
                "budget":400_000,"categories":["Alam","Kuliner","Wisata"]}},
    {"desc": "Multi-kategori 2 (Kuliner+Belanja)",
     "params": {"home":{"lat":-6.9145,"lng":107.6020},"homeName":"Stasiun Bandung",
                "count":4,"maxKm":20,"startMin":10*60,"endMin":21*60,
                "budget":300_000,"categories":["Kuliner","Belanja"]}},
    {"desc": "Single kategori (Alam)",
     "params": {"home":{"lat":-6.8126,"lng":107.6178},"homeName":"Lembang",
                "count":3,"maxKm":None,"startMin":7*60,"endMin":18*60,
                "budget":200_000,"categories":["Alam"]}},
    {"desc": "Semua 4 kategori",
     "params": {"home":{"lat":-6.9215,"lng":107.6071},"homeName":"Alun-Alun",
                "count":5,"maxKm":None,"startMin":8*60,"endMin":21*60,
                "budget":None,"categories":["Alam","Kuliner","Wisata","Belanja"]}},
]

all_pass = True
for tc in test_cases:
    res  = full_pipeline_test(tc["params"])
    cats_got = [s["dest"]["category"] for s in res.get("steps",[])]
    cats_req = tc["params"]["categories"]
    missing  = [c for c in cats_req if c not in cats_got]

    status = "✅" if not missing else "❌"
    if missing: all_pass = False

    print(f"{status} {tc['desc']}")
    print(f"   Destinasi : {[s['dest']['name'] for s in res.get('steps',[])]}")
    print(f"   Kategori  : {cats_got}")
    if missing:
        print(f"   ❌ MISSING: {missing}")
    else:
        # Cek variety (tidak ada 3 berturut-turut kategori sama)
        consec = max(
            (sum(1 for _ in g) for _, g in __import__('itertools').groupby(cats_got)),
            default=0
        )
        variety_ok = consec <= 2
        print(f"   Variety   : {'✅ max konsekutif=' + str(consec) if variety_ok else '⚠️  konsekutif=' + str(consec)}")
    print()

print("✅ Semua test passed!" if all_pass else "⚠️  Ada test yang perlu dicek")


🧪 Integration Test

  🔄 Swap: 'Bukit Moko' (Alam) → 'Floating Market Lembang' (Kuliner)
  🔄 Swap: 'Tebing Keraton' (Alam) → 'Shoes and bags shopping street' (Wisata)
✅ Multi-kategori 3 (Alam+Kuliner+Wisata)
   Destinasi : ['Shoes and bags shopping street', 'Floating Market Lembang', 'Tangkuban Perahu', 'Eurad Highland']
   Kategori  : ['Wisata', 'Kuliner', 'Alam', 'Alam']
   Variety   : ✅ max konsekutif=2

  🔄 Swap: 'Angkringan Si Meong' (Kuliner) → 'Pasar Baru Square' (Belanja)
✅ Multi-kategori 2 (Kuliner+Belanja)
   Destinasi : ['Pasar Baru Square', 'Talago Minang', 'Ranah Minang', 'SOBREMESA RESTO']
   Kategori  : ['Belanja', 'Kuliner', 'Kuliner', 'Kuliner']
   Variety   : ⚠️  konsekutif=3

✅ Single kategori (Alam)
   Destinasi : ['Tebing Keraton', 'Bukit Moko', 'Taman Pandawa']
   Kategori  : ['Alam', 'Alam', 'Alam']
   Variety   : ⚠️  konsekutif=3

  🔄 Swap: 'Bukit Moko' (Alam) → 'RM Alas Daun' (Kuliner)
  🔄 Swap: 'Tebing Keraton' (Alam) → 'Chinatown Bandung' (Wisata)
  🔄 Swap: 'E

## 📏 CELL 17 — Evaluasi Model

In [30]:
# ============================================================
# CELL 16 — Integration Test (Full Pipeline) + Edge Cases
# ============================================================
# FIX — alias nama optimizer
optimizer = route_optimizer
print("✅ optimizer siap")

def rl_select_destinations(env: BandungTravelEnv, agent: QLearningAgent,
                            params: dict) -> list:
    """Inference RL: greedy policy (epsilon=0). Tidak melakukan double-append —
    bergantung pada env.selected sebagai sumber kebenaran."""
    saved_eps = agent.epsilon
    agent.epsilon = 0.0
    try:
        state, candidates = env.reset(params)
        candidate_ids = [env.df.iloc[i]["id"] for i in candidates]
        done = False
        target = max(1, int(params.get("count", 4)))
        while not done and len(env.selected) < target:
            valid = env.get_valid_actions()
            if not valid:
                break
            action = agent.choose_action(state, valid, candidate_ids)
            if action < 0 or action >= len(env.candidates):
                break
            state, _, done, _ = env.step(action)
        # Pakai env.selected sebagai single source of truth
        return [env.df.iloc[i].to_dict() for i in env.selected]
    finally:
        agent.epsilon = saved_eps


def _smart_fallback_fill(selected: list, params: dict, target: int) -> list:
    """Lengkapi `selected` dengan kandidat CBF yang tetap menghormati
    sisa budget, sisa waktu, dan max_km dari home."""
    if len(selected) >= target:
        return selected
    home_lat = params["home"]["lat"]
    home_lng = params["home"]["lng"]
    spent = sum(int(d.get("ticket", 0)) for d in selected)
    used_dur = sum(int(d.get("duration", 0)) for d in selected)
    # Penting: budget=0 valid (artinya hanya gratis), bukan "no limit"
    has_budget = params.get("budget") is not None
    remaining_budget = (params["budget"] - spent) if has_budget else None
    remaining_time = max(0, params["endMin"] - params["startMin"] - used_dur)

    rec = cbf_model.recommend(
        categories=params.get("categories", []),
        budget=remaining_budget,
        max_km=params.get("maxKm"),
        home_lat=home_lat, home_lng=home_lng,
        top_n=30,
    )
    seen = {d["id"] for d in selected}
    for _, r in rec.iterrows():
        if len(selected) >= target:
            break
        if r["id"] in seen:
            continue
        if has_budget and int(r.get("ticket", 0)) > remaining_budget:
            continue
        if int(r.get("duration", 60)) > remaining_time:
            continue
        selected.append(r.to_dict())
        seen.add(r["id"])
        spent += int(r.get("ticket", 0))
        if has_budget:
            remaining_budget -= int(r.get("ticket", 0))
        remaining_time -= int(r.get("duration", 60))
    return selected


def full_pipeline_test(params: dict) -> dict:
    selected = rl_select_destinations(env, rl_agent, {
        "categories": params.get("categories", []),
        "budget": params.get("budget"),
        "max_km": params.get("maxKm"),
        "count": params.get("count", 4),
        "startMin": params["startMin"],
        "endMin": params["endMin"],
        "home_lat": params["home"]["lat"],
        "home_lng": params["home"]["lng"],
    })
    selected = _smart_fallback_fill(selected, params, params.get("count", 4))
    ordered = optimizer.nearest_neighbor_route(params["home"], selected)
    return optimizer.build_itinerary(
        home=params["home"],
        home_name=params["homeName"],
        ordered_destinations=ordered,
        start_min=params["startMin"],
        end_min=params["endMin"],
    )


def _print_itinerary(label: str, params: dict, res: dict):
    print(f"\n{'='*60}\n{label}\n{'='*60}")
    print(f"Destinasi terpilih: {len(res['steps'])}")
    for s in res["steps"]:
        a = f"{s['arriveAt']//60:02d}:{s['arriveAt']%60:02d}"
        print(f"  {s['idx']}. {s['dest']['name']:<28} ({s['dest']['category']}) — tiba {a}")
    print(f"💰 Total biaya: Rp {res['totalCost']:,}")
    print(f"📏 Total jarak: {res['totalKm']:.1f} km | totalTime {res['totalTime']} mnt")
    print(f"🏠 Tiba kembali: {res['arriveHome']//60:02d}:{res['arriveHome']%60:02d} | overTime: {res['overBudget']}")


# ---------- 3 skenario utama ----------
test_cases = [
    {"home": {"lat": -6.9215, "lng": 107.6071}, "homeName": "Alun-Alun Bandung",
     "count": 4, "maxKm": None, "startMin": 9 * 60, "endMin": 21 * 60,
     "budget": 500_000, "categories": ["Alam", "Kuliner"]},
    {"home": {"lat": -6.8126, "lng": 107.6178}, "homeName": "Pasar Lembang",
     "count": 3, "maxKm": 25, "startMin": 8 * 60, "endMin": 18 * 60,
     "budget": 200_000, "categories": ["Alam"]},
    {"home": {"lat": -6.9145, "lng": 107.6020}, "homeName": "Stasiun Bandung",
     "count": 5, "maxKm": None, "startMin": 10 * 60, "endMin": 20 * 60,
     "budget": None, "categories": ["Kuliner", "Budaya", "Belanja"]},
]
main_results = []
for i, p in enumerate(test_cases, 1):
    res = full_pipeline_test(p)
    _print_itinerary(f"TEST CASE {i}: {p['homeName']}", p, res)
    main_results.append(res)


# ---------- Edge cases ----------
print("\n" + "#" * 60)
print("# EDGE CASES")
print("#" * 60)

edge_cases = [
    # Edge: budget = 0 → hanya destinasi gratis (ticket=0)
    {"label": "Edge 1 — budget = 0 (hanya gratis)",
     "params": {"home": {"lat": -6.9215, "lng": 107.6071}, "homeName": "Alun-Alun",
                "count": 3, "maxKm": None, "startMin": 9 * 60, "endMin": 17 * 60,
                "budget": 0, "categories": ["Budaya", "Belanja"]}},
    # Edge: count > kandidat tersedia (count=8 + max_km kecil)
    {"label": "Edge 2 — count > kandidat (max_km 5km, count=8)",
     "params": {"home": {"lat": -6.9215, "lng": 107.6071}, "homeName": "Alun-Alun",
                "count": 8, "maxKm": 5, "startMin": 8 * 60, "endMin": 22 * 60,
                "budget": None, "categories": ["Kuliner", "Budaya"]}},
    # Edge: semua kategori dipilih
    {"label": "Edge 3 — semua kategori",
     "params": {"home": {"lat": -6.9215, "lng": 107.6071}, "homeName": "Alun-Alun",
                "count": 5, "maxKm": None, "startMin": 8 * 60, "endMin": 21 * 60,
                "budget": 500_000,
                "categories": ["Alam", "Kuliner", "Budaya", "Wisata", "Belanja"]}},
    # Edge: tidak ada kategori dipilih (frontend kadang kirim list kosong)
    {"label": "Edge 4 — tidak ada kategori (categories=[])",
     "params": {"home": {"lat": -6.9215, "lng": 107.6071}, "homeName": "Alun-Alun",
                "count": 3, "maxKm": None, "startMin": 9 * 60, "endMin": 19 * 60,
                "budget": 300_000, "categories": []}},
]
for ec in edge_cases:
    try:
        res = full_pipeline_test(ec["params"])
        _print_itinerary(ec["label"], ec["params"], res)
        # Validasi soft: minimal 1 destinasi terambil (kecuali jika benar-benar mustahil)
        if len(res["steps"]) == 0:
            print("  ℹ️  0 destinasi — kondisi memang mustahil, fallback kosong.")
    except Exception as e:
        print(f"  ❌ {ec['label']} crashed: {e}")


✅ optimizer siap

TEST CASE 1: Alun-Alun Bandung
Destinasi terpilih: 4
  1. Tebing Keraton               (Alam) — tiba 09:22
  2. Bukit Moko                   (Alam) — tiba 11:54
  3. Eurad Highland               (Alam) — tiba 14:08
  4. Tangkuban Perahu             (Alam) — tiba 17:15
💰 Total biaya: Rp 85,000
📏 Total jarak: 89.7 km | totalTime 694 mnt
🏠 Tiba kembali: 20:34 | overTime: False

TEST CASE 2: Pasar Lembang
Destinasi terpilih: 3
  1. Tebing Keraton               (Alam) — tiba 08:24
  2. Bukit Moko                   (Alam) — tiba 10:56
  3. Taman Piset                  (Alam) — tiba 13:23
💰 Total biaya: Rp 55,000
📏 Total jarak: 73.3 km | totalTime 461 mnt
🏠 Tiba kembali: 15:41 | overTime: False

TEST CASE 3: Stasiun Bandung
Destinasi terpilih: 5
  1. Talago Minang                (Kuliner) — tiba 10:07
  2. Resep Moyang                 (Kuliner) — tiba 11:34
  3. Angkringan Si Meong          (Kuliner) — tiba 13:01
  4. SOBREMESA RESTO              (Kuliner) — tiba 14:32
  5. 

## 💾 CELL 18 — Export & Summary

In [31]:
# ============================================================
# CELL 18 — Export & Summary + Sample API Response
# ============================================================
from datetime import date
print("\n" + "="*60)
print("📦 EXPORT SUMMARY — Bandung AI Travel Agent v2")
print("="*60)

sample_params = {
    "home": {"lat": -6.9215, "lng": 107.6071},
    "homeName": "Alun-Alun Bandung",
    "count": 4, "maxKm": None,
    "startMin": 9*60, "endMin": 21*60,
    "budget": 500_000,
    "categories": ["Alam", "Kuliner"],
}
sample_itinerary = full_pipeline_test(sample_params)
sample_response  = {
    **sample_itinerary,
    "story": {
        "story": "(akan di-generate oleh Notebook 02 LLM Storyteller)",
        "vibe":  "Alam & Kuliner",
    },
    "data_last_updated": str(date.today()),
}

with open("data/processed/sample_api_request.json","w",encoding="utf-8") as f:
    json.dump(sample_params, f, ensure_ascii=False, indent=2)
with open("data/processed/sample_api_response.json","w",encoding="utf-8") as f:
    json.dump(sample_response, f, ensure_ascii=False, indent=2, default=str)

# Validasi kontrak frontend
required = ["steps","totalCost","totalKm","totalTime","returnKm",
            "returnMin","arriveHome","overBudget","spareMin","story","data_last_updated"]
missing = [k for k in required if k not in sample_response]
print(f"  Kontrak frontend: {'✅ lengkap' if not missing else '❌ missing: '+str(missing)}")

files = [
    "data/processed/destinations.csv",
    "data/processed/feature_matrix.npy",
    "data/processed/sample_api_request.json",
    "data/processed/sample_api_response.json",
    "data/last_updated.txt",
    "models/cbf_model.pkl",
    "models/rl_agent.pkl",
    "models/scaler.pkl",
    "models/label_encoders.pkl",
]
print("\n📁 File output:")
for f in files:
    if os.path.exists(f):
        print(f"  ✅ {f} ({os.path.getsize(f)/1024:.1f} KB)")
    else:
        print(f"  ❌ MISSING: {f}")

print(f"\n✅ Kategori aktif: {CATEGORY_ORDER}")
print(f"✅ Dataset: {len(df_clean)} destinasi")
print(df_clean["category"].value_counts().to_dict())



📦 EXPORT SUMMARY — Bandung AI Travel Agent v2
  Kontrak frontend: ✅ lengkap

📁 File output:
  ✅ data/processed/destinations.csv (282.2 KB)
  ✅ data/processed/feature_matrix.npy (334.5 KB)
  ✅ data/processed/sample_api_request.json (0.2 KB)
  ✅ data/processed/sample_api_response.json (3.1 KB)
  ✅ data/last_updated.txt (0.0 KB)
  ✅ models/cbf_model.pkl (17425.4 KB)
  ✅ models/rl_agent.pkl (86.2 KB)
  ✅ models/scaler.pkl (0.7 KB)
  ✅ models/label_encoders.pkl (3.4 KB)

✅ Kategori aktif: ['Alam', 'Kuliner', 'Wisata', 'Belanja']
✅ Dataset: 1476 destinasi
{'Alam': 727, 'Kuliner': 646, 'Wisata': 61, 'Belanja': 42}


# ============================================================
# CELL 18 — Export & Summary + Sample API Response
# ============================================================
print("\n" + "=" * 60)
print("📦 EXPORT SUMMARY — Bandung AI Travel Agent")
print("=" * 60)

# ---------- Sample API request + response (sesuai kontrak frontend) ----------
sample_params = {
    "home": {"lat": -6.9215, "lng": 107.6071},
    "homeName": "Alun-Alun Bandung",
    "count": 4,
    "maxKm": None,
    "startMin": 9 * 60,
    "endMin": 21 * 60,
    "budget": 500_000,
    "categories": ["Alam", "Kuliner"],
}
sample_itinerary = full_pipeline_test(sample_params)
sample_response = {
    **sample_itinerary,
    # story biasanya di-generate di Notebook 02 — di sini placeholder kosong
    # supaya backend developer punya gambaran shape lengkap.
    "story": {
        "intro": "(akan di-generate oleh Notebook 02 / Groq API)",
        "highlights": ["(per destinasi)"] * len(sample_itinerary["steps"]),
        "tips": ["(2-4 tips praktis)"],
        "closing": "(1 kalimat penutup)",
        "vibe": "Alam & Kuliner",
    },
    "data_last_updated": str(date.today()),
}

with open("data/processed/sample_api_request.json", "w", encoding="utf-8") as f:
    json.dump(sample_params, f, ensure_ascii=False, indent=2)
with open("data/processed/sample_api_response.json", "w", encoding="utf-8") as f:
    json.dump(sample_response, f, ensure_ascii=False, indent=2)
print("✅ data/processed/sample_api_request.json")
print("✅ data/processed/sample_api_response.json (story = placeholder)")

# ---------- Validasi kontrak frontend (basic) ----------
required_root = ["steps", "totalCost", "totalKm", "totalTime", "returnKm",
                  "returnMin", "arriveHome", "overBudget", "spareMin",
                  "story", "data_last_updated"]
missing = [k for k in required_root if k not in sample_response]
print(f"  Kontrak frontend (root keys): {'✅ lengkap' if not missing else '❌ missing: ' + str(missing)}")

# ---------- Cek file model ----------
files_to_check = [
    "data/processed/destinations.csv",
    "data/processed/feature_matrix.npy",
    "data/processed/sample_api_request.json",
    "data/processed/sample_api_response.json",
    "data/last_updated.txt",
    "models/cbf_model.pkl",
    "models/rl_agent.pkl",
    "models/scaler.pkl",
    "models/label_encoders.pkl",
]
for f in files_to_check:
    if os.path.exists(f):
        size = os.path.getsize(f) / 1024
        print(f"  ✅ {f} ({size:.1f} KB)")
    else:
        print(f"  ❌ {f} — MISSING!")

df_summary = pd.read_csv("data/processed/destinations.csv")
print(f"\n📊 Dataset Summary:")
print(f"  Total destinasi: {len(df_summary)}")
print(df_summary.groupby("category").agg({
    "name": "count", "rating": "mean", "ticket": "median"
}).round(2))

print(f"\n⏰ Last Updated: {date.today()}")
print("\n✅ Semua model dan data siap digunakan oleh Notebook 2 dan Backend!")


## 📦 CELL 20 — Zip & Download

In [32]:
# ============================================================
# CELL 20 — Zip & Download
# ============================================================
import zipfile
from pathlib import Path

SOURCE_DIR = Path("/kaggle/working/bandung_travel_export")
SOURCE_DIR.mkdir(exist_ok=True)

# Copy semua output ke export folder
import shutil
(SOURCE_DIR/"models").mkdir(exist_ok=True)
(SOURCE_DIR/"data/processed").mkdir(parents=True, exist_ok=True)

for f in ["cbf_model.pkl","rl_agent.pkl","label_encoders.pkl","scaler.pkl"]:
    src = Path(f"models/{f}")
    if src.exists(): shutil.copy2(src, SOURCE_DIR/"models"/f)

for f in ["destinations.csv","feature_matrix.npy",
          "sample_api_request.json","sample_api_response.json"]:
    src = Path(f"data/processed/{f}")
    if src.exists(): shutil.copy2(src, SOURCE_DIR/"data/processed"/f)

src_lu = Path("data/last_updated.txt")
if src_lu.exists(): shutil.copy2(src_lu, SOURCE_DIR/"data/last_updated.txt")

# Zip ke /kaggle/working/ (BUKAN ke input yang read-only)
ZIP_PATH = Path("/kaggle/working/RS_TRAIN_v2.zip")
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for file in sorted(SOURCE_DIR.rglob("*")):
        if file.is_file():
            zf.write(file, file.relative_to(SOURCE_DIR.parent))
            print(f"  + {file.relative_to(SOURCE_DIR.parent)}")

size_mb = ZIP_PATH.stat().st_size / 1024 / 1024
print(f"\n✅ {ZIP_PATH} ({size_mb:.1f} MB)")
print("\n📥 Download: panel kanan → Output → RS_TRAIN_v2.zip → Download")
print("   Atau: Save Version → tab Output → download")


  + bandung_travel_export/data/last_updated.txt
  + bandung_travel_export/data/processed/destinations.csv
  + bandung_travel_export/data/processed/feature_matrix.npy
  + bandung_travel_export/data/processed/sample_api_request.json
  + bandung_travel_export/data/processed/sample_api_response.json
  + bandung_travel_export/models/cbf_model.pkl
  + bandung_travel_export/models/label_encoders.pkl
  + bandung_travel_export/models/rl_agent.pkl
  + bandung_travel_export/models/scaler.pkl

✅ /kaggle/working/RS_TRAIN_v2.zip (15.4 MB)

📥 Download: panel kanan → Output → RS_TRAIN_v2.zip → Download
   Atau: Save Version → tab Output → download
